In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    brier_score_loss
)

In [ ]:
# ============================================================
# RUNTIME BENCHMARK LOGGER
# ============================================================

from time import perf_counter
from datetime import datetime
from pathlib import Path
import pandas as pd
import os

RUNTIME_FILE = Path("FINAL_experimental_runtime_summary.csv")

if RUNTIME_FILE.exists():
    RUNTIME_LOG = pd.read_csv(RUNTIME_FILE).to_dict("records")
else:
    RUNTIME_LOG = []


def start_timer(experiment_name):
    print("\n" + "=" * 75)
    print(f"STARTING: {experiment_name}")
    print("Clock time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("=" * 75)

    return perf_counter()


def stop_timer(experiment_name, start_time):
    elapsed_sec = perf_counter() - start_time

    record = {
        "experiment": experiment_name,
        "runtime_seconds": elapsed_sec,
        "runtime_minutes": elapsed_sec / 60.0,
        "runtime_hours": elapsed_sec / 3600.0,
    }

    RUNTIME_LOG.append(record)

    pd.DataFrame(RUNTIME_LOG).to_csv(
        RUNTIME_FILE,
        index=False
    )

    print("\n" + "-" * 75)
    print(f"FINISHED: {experiment_name}")
    print(f"Runtime: {elapsed_sec:.2f} seconds")
    print(f"Runtime: {elapsed_sec / 60.0:.4f} minutes")
    print("-" * 75)

    return elapsed_sec

In [ ]:
# ============================================================
# REPRODUCIBILITY AUDIT — SOFTWARE AND HARDWARE ENVIRONMENT
# Run inside BASIC_SLICE_PRELIM_env
# ============================================================

import sys
import os
import platform
import subprocess
import importlib
import pandas as pd

print("=" * 75)
print("EXPERIMENTAL COMPUTING ENVIRONMENT")
print("=" * 75)

# ------------------------------------------------------------
# 1. Python / operating system
# ------------------------------------------------------------
print("\n[PYTHON]")
print("Python version :", sys.version.replace("\n", " "))
print("Executable     :", sys.executable)

print("\n[OPERATING SYSTEM]")
print("OS             :", platform.system())
print("OS release     :", platform.release())
print("OS version     :", platform.version())
print("Architecture   :", platform.machine())
print("Platform       :", platform.platform())

# ------------------------------------------------------------
# 2. CPU
# ------------------------------------------------------------
print("\n[PROCESSOR]")

cpu_name = platform.processor()

# Windows fallback: platform.processor() can occasionally be blank
if not cpu_name and platform.system() == "Windows":
    try:
        result = subprocess.run(
            [
                "powershell",
                "-Command",
                "(Get-CimInstance Win32_Processor).Name"
            ],
            capture_output=True,
            text=True
        )
        cpu_name = result.stdout.strip()
    except Exception:
        pass

print("Processor      :", cpu_name if cpu_name else "Unable to determine")
print("Logical CPUs   :", os.cpu_count())

# ------------------------------------------------------------
# 3. Memory
# ------------------------------------------------------------
print("\n[MEMORY]")

try:
    import psutil

    vm = psutil.virtual_memory()

    print(
        "Installed RAM :",
        f"{vm.total / (1024**3):.2f} GB"
    )
except Exception as e:
    print("Installed RAM : psutil unavailable:", e)

# ------------------------------------------------------------
# 4. Important package versions
# ------------------------------------------------------------
packages = [
    "numpy",
    "pandas",
    "scipy",
    "sklearn",
    "matplotlib",
    "openpyxl",
    "joblib",
    "statsmodels",
    "upsetplot",
    "jupyter",
    "notebook",
    "ipykernel",
    "psutil",
]

rows = []

for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "version attribute unavailable")
    except Exception:
        version = "not installed / not importable"

    rows.append({
        "package": pkg,
        "version": version
    })

versions_df = pd.DataFrame(rows)

print("\n[CORE PACKAGE VERSIONS]")
print(versions_df.to_string(index=False))

# ------------------------------------------------------------
# 5. Save reproducibility record
# ------------------------------------------------------------
versions_df.to_csv(
    "experimental_package_versions.csv",
    index=False
)

with open(
    "experimental_environment.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("EXPERIMENTAL COMPUTING ENVIRONMENT\n")
    f.write("=" * 70 + "\n")

    f.write(f"Python: {sys.version}\n")
    f.write(f"Executable: {sys.executable}\n")
    f.write(f"OS: {platform.platform()}\n")
    f.write(f"Architecture: {platform.machine()}\n")
    f.write(f"Processor: {cpu_name}\n")
    f.write(f"Logical CPUs: {os.cpu_count()}\n")

    try:
        f.write(
            f"Installed RAM: "
            f"{psutil.virtual_memory().total / (1024**3):.2f} GB\n"
        )
    except Exception:
        pass

print("\nSaved:")
print("  experimental_package_versions.csv")
print("  experimental_environment.txt")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# LOAD NOTEBOOK 8 OUTPUTS
# ============================================================

WORK_DIR = Path.cwd()

required_files = {
    "dataset": WORK_DIR / "df_with_diagnostics.csv",
    "slice harm": WORK_DIR / "slice_harm.csv",
    "slice membership": WORK_DIR / "slice_membership.csv",
    "ASAP audit pool":
        WORK_DIR / "nb8_asap_complete_candidate_audit.csv",
    "ASAP ablation pool":
        WORK_DIR / "nb8_asap_candidate_pool_with_ablations.csv",
}

for label, path in required_files.items():
    print(
        f"{label}:",
        path,
        "FOUND" if path.exists() else "MISSING"
    )

missing_files = [
    str(path)
    for path in required_files.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Run Notebook 8 and create these files first:\n"
        + "\n".join(missing_files)
    )

df = pd.read_csv(
    required_files["dataset"]
)

slice_harm = pd.read_csv(
    required_files["slice harm"],
    index_col=0
).squeeze("columns")

slice_membership = pd.read_csv(
    required_files["slice membership"]
)

# This is the governed ASAP pool created in Notebook 8.
asap_pool = pd.read_csv(
    required_files["ASAP audit pool"]
)

# This contains the full-ASAP and guardrail-ablation indicators.
asap_ablation_pool = pd.read_csv(
    required_files["ASAP ablation pool"]
)

# Clean names
df.columns = df.columns.astype(str).str.strip()
slice_membership.columns = (
    slice_membership.columns.astype(str).str.strip()
)
asap_pool.columns = (
    asap_pool.columns.astype(str).str.strip()
)
asap_ablation_pool.columns = (
    asap_ablation_pool.columns.astype(str).str.strip()
)

required_asap_columns = [
    "slice_name",
    "asap_eligible_full",
]

missing_asap_columns = [
    column
    for column in required_asap_columns
    if column not in asap_pool.columns
]

if missing_asap_columns:
    raise KeyError(
        "The Notebook 8 ASAP audit file is missing: "
        f"{missing_asap_columns}. "
        f"Available columns: {asap_pool.columns.tolist()}"
    )

asap_pool["slice_name"] = (
    asap_pool["slice_name"]
    .astype(str)
    .str.strip()
)

asap_pool["asap_eligible_full"] = (
    pd.to_numeric(
        asap_pool["asap_eligible_full"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

print("\nLoaded successfully:")
print("df:", df.shape)
print("slice_harm:", slice_harm.shape)
print("slice_membership:", slice_membership.shape)
print("asap_pool:", asap_pool.shape)
print(
    "Full-ASAP eligible slices:",
    int(asap_pool["asap_eligible_full"].sum())
)

In [ ]:
#import pandas as pd
#
## Load dataset
#df = pd.read_csv("df_with_diagnostics.csv")
#
## Load slice harm
#slice_harm = pd.read_csv("slice_harm.csv", index_col=0).squeeze()
#
## Load slice membership
#slice_membership = pd.read_csv("slice_membership.csv")
#
#print("df loaded:", df.shape)
#print("slice_harm loaded:", slice_harm.shape)
#print("slice_membership loaded:", slice_membership.shape)

In [ ]:
print(slice_harm.sort_values(ascending=False).head())
print(slice_membership.columns[:5])

In [ ]:
required_objects = ["df", "slice_harm", "slice_membership"]

for obj in required_objects:
    print(obj, "exists:", obj in globals())

Define target and features

In [ ]:
target_col = "target" if "target" in df.columns else "y"

exclude_cols = [
    target_col,
    "oof_logloss", "HCC", "FN",
    "p_harm_weighted",
    "w_slice", "w_suspicious", "w_final"
]

feature_cols = [
    c for c in df.columns
    if c not in exclude_cols
    and not c.startswith("SLICE__")
]

X = df[feature_cols].copy()
y = df[target_col].values

print("Target:", target_col)
print("Number of features:", len(feature_cols))

Normalize positive slice harm only

In [ ]:
positive_harm = slice_harm[slice_harm > 0].copy()

harm_norm = (positive_harm - positive_harm.min()) / (
    positive_harm.max() - positive_harm.min()
)

harm_norm = harm_norm.fillna(0)

harm_table = pd.DataFrame({
    "slice": harm_norm.index,
    "raw_harm": positive_harm.loc[harm_norm.index],
    "harm_norm": harm_norm.values
}).sort_values("raw_harm", ascending=False)

harm_table.head(15)

Identify top-k harmful slices

In [ ]:
TOP_K = 5

top_slices = harm_table.head(TOP_K)["slice"].tolist()

print("Top slices:")
for s in top_slices:
    print("-", s)

Helper: weight normalization

In [ ]:
def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    return w / np.mean(w)

Repair arm A: Top-k harm-slice weighting

In [ ]:
def build_topk_weights(slice_membership, harm_norm, top_slices, lambda_=1.5):
    w = np.ones(len(slice_membership))

    for s in top_slices:
        if s not in slice_membership.columns:
            continue
        if s not in harm_norm.index:
            continue

        mask = slice_membership[s].values == 1
        w[mask] += lambda_ * harm_norm.loc[s]

    return normalize_weights(w)

df["w_topk"] = build_topk_weights(
    slice_membership=slice_membership,
    harm_norm=harm_norm,
    top_slices=top_slices,
    lambda_=1.5
)

df["w_topk"].describe()

Check upweighted percentage

In [ ]:
print("Percent upweighted:", 100 * (df["w_topk"] > 1.001).mean())

Repair arm B: Squared-harm weighting

In [ ]:
def build_squared_harm_weights(slice_membership, harm_norm, lambda_=2.0):
    w = np.ones(len(slice_membership))

    for s in harm_norm.index:
        if s not in slice_membership.columns:
            continue

        mask = slice_membership[s].values == 1
        sharp_harm = harm_norm.loc[s] ** 2
        w[mask] += lambda_ * sharp_harm

    return normalize_weights(w)

df["w_squared_harm"] = build_squared_harm_weights(
    slice_membership=slice_membership,
    harm_norm=harm_norm,
    lambda_=2.0
)

df["w_squared_harm"].describe()

In [ ]:
print("Percent upweighted:", 100 * (df["w_squared_harm"] > 1.001).mean())

Repair arm C: Worst-slice targeted boost

In [ ]:
worst_slice_target = "abdominal_pain_and_hr_missing"

def build_worst_slice_boost(slice_membership, worst_slice, boost=2.5):
    w = np.ones(len(slice_membership))

    if worst_slice in slice_membership.columns:
        mask = slice_membership[worst_slice].values == 1
        w[mask] += boost
    else:
        print("WARNING: worst slice not found:", worst_slice)

    return normalize_weights(w)

df["w_worst_boost"] = build_worst_slice_boost(
    slice_membership=slice_membership,
    worst_slice=worst_slice_target,
    boost=2.5
)

df["w_worst_boost"].describe()

In [ ]:
print("Percent upweighted:", 100 * (df["w_worst_boost"] > 1.001).mean())

Repair arm D: Combined targeted repair

In [ ]:
def build_combined_targeted_weights(
    slice_membership,
    harm_norm,
    top_slices,
    worst_slice,
    lambda_topk=1.2,
    lambda_squared=1.5,
    worst_boost=2.0
):
    w = np.ones(len(slice_membership))

    # top-k repair
    for s in top_slices:
        if s in slice_membership.columns and s in harm_norm.index:
            mask = slice_membership[s].values == 1
            w[mask] += lambda_topk * harm_norm.loc[s]

    # squared-harm repair
    for s in harm_norm.index:
        if s in slice_membership.columns:
            mask = slice_membership[s].values == 1
            w[mask] += lambda_squared * (harm_norm.loc[s] ** 2)

    # worst-slice targeted repair
    if worst_slice in slice_membership.columns:
        mask = slice_membership[worst_slice].values == 1
        w[mask] += worst_boost

    return normalize_weights(w)

df["w_combined_targeted"] = build_combined_targeted_weights(
    slice_membership=slice_membership,
    harm_norm=harm_norm,
    top_slices=top_slices,
    worst_slice=worst_slice_target,
    lambda_topk=1.2,
    lambda_squared=1.5,
    worst_boost=2.0
)

df["w_combined_targeted"].describe()

In [ ]:
print("Percent upweighted:", 100 * (df["w_combined_targeted"] > 1.001).mean())

Suspicious-label downweighting, targeted version

In [ ]:
def build_targeted_suspicious_downweight(
    df,
    slice_membership,
    top_slices,
    rho=0.6,
    loss_quantile=0.90
):
    in_top_slice = np.zeros(len(df), dtype=bool)

    for s in top_slices:
        if s in slice_membership.columns:
            in_top_slice |= (slice_membership[s].values == 1)

    high_loss = df["oof_logloss"] > df["oof_logloss"].quantile(loss_quantile)
    hcc = df["high_conf_contradiction"] == 1

    suspicious = in_top_slice & (high_loss | hcc)

    d = np.ones(len(df))
    d[suspicious] = rho

    return d, suspicious

df["w_suspicious_targeted"], df["suspicious_targeted_flag"] = build_targeted_suspicious_downweight(
    df=df,
    slice_membership=slice_membership,
    top_slices=top_slices,
    rho=0.6,
    loss_quantile=0.90
)

print("Suspicious targeted percent:", 100 * df["suspicious_targeted_flag"].mean())

Final combined targeted + suspicious repair

In [ ]:
df["w_combined_targeted_suspicious"] = normalize_weights(
    df["w_combined_targeted"].values * df["w_suspicious_targeted"].values
)

df["w_combined_targeted_suspicious"].describe()

In [ ]:
print(
    "Percent changed from 1:",
    100 * (np.abs(df["w_combined_targeted_suspicious"] - 1) > 0.001).mean()
)

In [ ]:
# ============================================================
# SOURCE-SPECIFIC REPAIR WEIGHTS
# Governed positive-harm slices only
# ============================================================

import os
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Load clean slice-name lists
# ------------------------------------------------------------

def load_slice_list(
    paths,
    required=False,
):
    """
    Load a clean and unique slice-name list.
    """
    for path in paths:
        if not os.path.exists(path):
            continue

        table = pd.read_csv(path)

        if "slice_name" not in table.columns:
            raise KeyError(
                f"{path} does not contain a slice_name column."
            )

        slice_names = (
            table["slice_name"]
            .dropna()
            .astype(str)
            .str.strip()
        )

        slice_names = [
            slice_name
            for slice_name in slice_names
            if slice_name
        ]

        slice_names = list(
            dict.fromkeys(slice_names)
        )

        if required and not slice_names:
            raise ValueError(
                f"{path} exists but contains no valid slices."
            )

        return slice_names, path

    if required:
        raise FileNotFoundError(
            f"None of the required files were found: {paths}"
        )

    print(
        "WARNING: no file found:",
        paths,
    )

    return [], None


clinical_discovery_slices, clinical_file = (
    load_slice_list([
        "nb8_clinical_discovery_slices.csv"
    ])
)

error_guided_discovery_slices, error_file = (
    load_slice_list([
        "nb8_error_guided_discovery_slices.csv"
    ])
)

dual_supported_slices, dual_supported_file = (
    load_slice_list([
        "nb8_dual_supported_slices.csv"
    ])
)

dual_source_combined_slices, combined_file = (
    load_slice_list([
        "nb8_dual_source_combined_slices.csv"
    ])
)

repair_eligible_combined_slices, repair_eligible_file = (
    load_slice_list(
        [
            "nb8_repair_eligible_combined_slices.csv"
        ],
        required=True,
    )
)

monitoring_only_slices, monitoring_file = (
    load_slice_list([
        "nb8_monitoring_only_slices.csv"
    ])
)

top_ranked_repair_slices, ranked_file = (
    load_slice_list([
        "nb8_top_ranked_repair_slices.csv"
    ])
)

source_weight_table = pd.read_csv(
    "nb8_source_weight_table.csv"
)

if "slice_name" not in source_weight_table.columns:
    raise KeyError(
        "nb8_source_weight_table.csv does not contain "
        "a slice_name column."
    )

source_weight_table["slice_name"] = (
    source_weight_table["slice_name"]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# 2. Define the weighting function before using it
# ------------------------------------------------------------

def build_source_arm_weights(
    slice_membership,
    source_weight_table,
    slice_names,
    lambda_=1.5,
    use_combined_multiplier=False,
    max_weight=5.0,
):
    """
    Construct patient-level weights from governed slices.
    """
    weights = np.ones(
        len(slice_membership),
        dtype=float,
    )

    weight_lookup = (
        source_weight_table
        .drop_duplicates("slice_name")
        .set_index("slice_name")
    )

    used = []
    missing = []

    for slice_name in slice_names:

        if slice_name not in slice_membership.columns:
            missing.append(slice_name)
            continue

        if slice_name not in weight_lookup.index:
            missing.append(slice_name)
            continue

        harm_value = float(
            weight_lookup.loc[
                slice_name,
                "harm_norm_for_repair",
            ]
        )

        if harm_value <= 0:
            continue

        if use_combined_multiplier:
            multiplier = float(
                weight_lookup.loc[
                    slice_name,
                    "combined_evidence_multiplier",
                ]
            )
        else:
            multiplier = 1.0

        membership_mask = (
            slice_membership[slice_name]
            .astype(bool)
            .to_numpy()
        )

        weights[membership_mask] += (
            lambda_
            * harm_value
            * multiplier
        )

        used.append(slice_name)

    weights = np.clip(
        weights,
        1.0,
        max_weight,
    )

    return weights, used, missing


# ------------------------------------------------------------
# 3. Apply the same repair-eligibility rule to every source arm
# ------------------------------------------------------------

repair_eligible_set = set(
    repair_eligible_combined_slices
)

clinical_repair_slices = sorted(
    set(clinical_discovery_slices)
    & repair_eligible_set
)

error_guided_repair_slices = sorted(
    set(error_guided_discovery_slices)
    & repair_eligible_set
)

dual_supported_repair_slices = sorted(
    set(dual_supported_slices)
    & repair_eligible_set
)

combined_repair_slices = sorted(
    repair_eligible_set
)


# ------------------------------------------------------------
# 4. Validate source-specific repair lists
# ------------------------------------------------------------

membership_slice_names = set(
    slice_membership.columns
    .astype(str)
)

weight_table_slice_names = set(
    source_weight_table["slice_name"]
)

source_list_audit = {
    "clinical_repair": clinical_repair_slices,
    "error_guided_repair": error_guided_repair_slices,
    "dual_supported_repair": dual_supported_repair_slices,
    "combined_repair": combined_repair_slices,
}

for source_name, slice_names in source_list_audit.items():

    missing_membership = [
        slice_name
        for slice_name in slice_names
        if slice_name not in membership_slice_names
    ]

    missing_weights = [
        slice_name
        for slice_name in slice_names
        if slice_name not in weight_table_slice_names
    ]

    print(
        source_name,
        "| slices:",
        len(slice_names),
        "| missing membership:",
        missing_membership,
        "| missing weight rows:",
        missing_weights,
    )

    if missing_membership:
        raise ValueError(
            f"{source_name} contains slices missing from "
            f"slice_membership: {missing_membership}"
        )

    if missing_weights:
        raise ValueError(
            f"{source_name} contains slices missing from "
            f"source_weight_table: {missing_weights}"
        )


# ------------------------------------------------------------
# 5. Build source-specific repair weights once
# ------------------------------------------------------------

df[
    "w_clinical_discovery"
], used_clinical, missing_clinical = (
    build_source_arm_weights(
        slice_membership=slice_membership,
        source_weight_table=source_weight_table,
        slice_names=clinical_repair_slices,
        lambda_=1.5,
        use_combined_multiplier=False,
    )
)

df[
    "w_error_guided_discovery"
], used_error, missing_error = (
    build_source_arm_weights(
        slice_membership=slice_membership,
        source_weight_table=source_weight_table,
        slice_names=error_guided_repair_slices,
        lambda_=1.5,
        use_combined_multiplier=False,
    )
)

df[
    "w_dual_supported"
], used_dual, missing_dual = (
    build_source_arm_weights(
        slice_membership=slice_membership,
        source_weight_table=source_weight_table,
        slice_names=dual_supported_repair_slices,
        lambda_=1.5,
        use_combined_multiplier=True,
    )
)

df[
    "w_dual_source_combined"
], used_combined, missing_combined = (
    build_source_arm_weights(
        slice_membership=slice_membership,
        source_weight_table=source_weight_table,
        slice_names=combined_repair_slices,
        lambda_=1.5,
        use_combined_multiplier=True,
    )
)

df[
    "w_basic_slice_ranked"
], used_ranked, missing_ranked = (
    build_source_arm_weights(
        slice_membership=slice_membership,
        source_weight_table=source_weight_table,
        slice_names=top_ranked_repair_slices,
        lambda_=1.5,
        use_combined_multiplier=True,
    )
)


# ------------------------------------------------------------
# 6. Validate that every repair arm changes some weights
# ------------------------------------------------------------

source_weight_columns = {
    "clinical_discovery_repair":
        "w_clinical_discovery",

    "error_guided_discovery_repair":
        "w_error_guided_discovery",

    "dual_supported_repair":
        "w_dual_supported",

    "dual_source_combined_discovery_repair":
        "w_dual_source_combined",
}

source_weight_audit_rows = []

for arm_name, weight_column in (
    source_weight_columns.items()
):
    weights = (
        df[weight_column]
        .to_numpy(dtype=float)
    )

    changed_mask = (
        ~np.isclose(weights, 1.0)
    )

    rows_changed = int(
        changed_mask.sum()
    )

    percent_changed = float(
        100.0
        * changed_mask.mean()
    )

    source_weight_audit_rows.append({
        "arm": arm_name,
        "weight_column": weight_column,
        "rows_changed": rows_changed,
        "percent_changed": percent_changed,
        "mean_weight": float(weights.mean()),
        "min_weight": float(weights.min()),
        "max_weight": float(weights.max()),
    })

    if rows_changed == 0:
        raise ValueError(
            f"{arm_name} is a no-op weight vector."
        )

source_weight_audit = pd.DataFrame(
    source_weight_audit_rows
)

display(source_weight_audit)


# ------------------------------------------------------------
# 7. Report actual slices used
# ------------------------------------------------------------

print(
    "\nClinical repair-eligible slices:",
    len(used_clinical),
    used_clinical,
)

print(
    "\nError-guided repair-eligible slices:",
    len(used_error),
    used_error,
)

print(
    "\nDual-supported repair-eligible slices:",
    len(used_dual),
    used_dual,
)

print(
    "\nCombined repair-eligible slices:",
    len(used_combined),
    used_combined,
)

print(
    "\nRanked repair slices:",
    len(used_ranked),
    used_ranked,
)

print(
    "\nMonitoring-only slices excluded from repair:",
    len(monitoring_only_slices),
    monitoring_only_slices,
)

TOP_K sensitivity

## TOP_K Sensitivity for BASiC-Slice Ranker

In [ ]:
# ============================================================
# TOP_K SENSITIVITY FOR BASiC-SLICE RANKER
# Tests whether ranked repair works better with K = 5, 7, or 10
# ============================================================

rank_table = pd.read_csv("nb8_basic_slice_rank_table.csv")

TOP_K_VALUES = [5, 7, 10]

topk_ranked_slices = {}

for k in TOP_K_VALUES:
    selected = (
        rank_table[
            (rank_table["nb8_harm"] > 0)
            & (rank_table["artifact_risk"] == 0)
        ]
        .sort_values("basic_slice_rank_score", ascending=False)
        .head(k)["slice_name"]
        .tolist()
    )

    topk_ranked_slices[k] = selected

    print(f"\nTOP_K = {k}")
    print(selected)

    weight_col = f"w_basic_slice_ranked_top{k}"

    df[weight_col], used_k, missing_k = build_source_arm_weights(
        slice_membership=slice_membership,
        source_weight_table=source_weight_table,
        slice_names=selected,
        lambda_=1.5,
        use_combined_multiplier=True,
    )

    print("Used:", len(used_k), "Missing:", missing_k)
    print(df[weight_col].describe())

In [ ]:
# ============================================================
# OFFICIAL BASiC-SLICE RANKER SETTING
# Based on sensitivity analysis, TOP_K=10 is selected.
# ============================================================

OFFICIAL_TOP_K = 10

official_ranked_slices = topk_ranked_slices[OFFICIAL_TOP_K]

print("Official BASiC-Slice Ranker setting:")
print("TOP_K =", OFFICIAL_TOP_K)
print("Selected slices:", official_ranked_slices)

df["w_basic_slice_ranked_official"], used_official, missing_official = build_source_arm_weights(
    slice_membership=slice_membership,
    source_weight_table=source_weight_table,
    slice_names=official_ranked_slices,
    lambda_=1.5,
    use_combined_multiplier=True,
)

print("Used official slices:", len(used_official))
print("Missing official slices:", missing_official)

df[["w_basic_slice_ranked_official"]].describe()

# New ranker Ek = H + D + S. Start**************************************************

In [ ]:
# ============================================================
# REVISED ASAP-CONSTRAINED BASiC REPAIR UTILITIES
# ============================================================

import ast
import numpy as np
import pandas as pd

revised_rank_table = pd.read_csv(
    "nb8_asap_constrained_basic_rank_table.csv"
)

revised_top10 = pd.read_csv(
    "nb8_asap_constrained_basic_top10.csv"
)["slice_name"].dropna().astype(str).tolist()

legacy_rank_table = pd.read_csv(
    "nb8_basic_slice_rank_table.csv"
)

ablation_top10_long = pd.read_csv(
    "nb8_asap_guardrail_ablation_top10.csv"
)

# Harm lookup.
harm_lookup = (
    revised_rank_table
    .set_index("slice_name")["harm_norm"]
    .to_dict()
)

provenance_lookup = (
    revised_rank_table
    .set_index("slice_name")[
        "dual_source_evidence"
    ]
    .to_dict()
)

def build_ranked_sample_weights(
    selected_slices,
    use_provenance_multiplier=False,
    lambda_repair=1.5,
    dual_multiplier=1.5,
    minimum_weight=1.0,
    maximum_weight=5.0,
):
    """
    Build patient-level repair weights.

    Ranking has already selected the slices.
    This function controls whether provenance also changes
    repair intensity.
    """

    weights = np.ones(
        len(df),
        dtype=float
    )

    for slice_name in selected_slices:
        if slice_name not in slice_membership.columns:
            print(
                f"Warning: {slice_name} not found "
                "in slice_membership."
            )
            continue

        harm_value = float(
            harm_lookup.get(slice_name, 0.0)
        )

        if harm_value <= 0:
            continue

        intensity = harm_value

        if use_provenance_multiplier:
            if int(
                provenance_lookup.get(slice_name, 0)
            ) == 1:
                intensity *= dual_multiplier

        weights += (
            lambda_repair
            * intensity
            * slice_membership[slice_name]
            .astype(float)
            .values
        )

    return np.clip(
        weights,
        minimum_weight,
        maximum_weight
    )

In [ ]:
# ============================================================
# EXPERIMENT 3
# WHERE SHOULD PROVENANCE ACT?
# Ranking, repair intensity, both, or neither
# ============================================================

# ------------------------------------------------------------
# Variant P1:
# provenance affects RANKING only
# ------------------------------------------------------------

p1_selected_slices = revised_top10

df["w_revised_P1_provenance_rank_only"] = (
    build_ranked_sample_weights(
        selected_slices=p1_selected_slices,
        use_provenance_multiplier=False,
        lambda_repair=1.5,
        dual_multiplier=1.5,
    )
)

# ------------------------------------------------------------
# Construct no-provenance ranking for P2/P4
# Ranking score uses harm + support only.
# Renormalize 0.60 and 0.10 to sum to one:
# harm = 0.60 / 0.70
# support = 0.10 / 0.70
# ------------------------------------------------------------

no_provenance_rank = revised_rank_table.copy()

W_HARM_NO_PROV = 0.60 / 0.70
W_SUPPORT_NO_PROV = 0.10 / 0.70

no_provenance_rank[
    "no_provenance_rank_score"
] = (
    W_HARM_NO_PROV
    * no_provenance_rank["harm_norm"]
    +
    W_SUPPORT_NO_PROV
    * no_provenance_rank[
        "support_norm_ranker"
    ]
)

no_provenance_rank = no_provenance_rank.sort_values(
    "no_provenance_rank_score",
    ascending=False
)

no_provenance_top10 = (
    no_provenance_rank
    .head(10)["slice_name"]
    .astype(str)
    .tolist()
)

# ------------------------------------------------------------
# Variant P2:
# provenance affects REPAIR INTENSITY only
# ------------------------------------------------------------

p2_selected_slices = no_provenance_top10

df["w_revised_P2_provenance_weight_only"] = (
    build_ranked_sample_weights(
        selected_slices=p2_selected_slices,
        use_provenance_multiplier=True,
        lambda_repair=1.5,
        dual_multiplier=1.5,
    )
)

# ------------------------------------------------------------
# Variant P3:
# provenance affects BOTH ranking and repair intensity
# ------------------------------------------------------------

p3_selected_slices = revised_top10

df["w_revised_P3_provenance_both"] = (
    build_ranked_sample_weights(
        selected_slices=p3_selected_slices,
        use_provenance_multiplier=True,
        lambda_repair=1.5,
        dual_multiplier=1.5,
    )
)

# ------------------------------------------------------------
# Variant P4:
# no provenance in ranking or repair intensity
# ------------------------------------------------------------

p4_selected_slices = no_provenance_top10

df["w_revised_P4_no_provenance"] = (
    build_ranked_sample_weights(
        selected_slices=p4_selected_slices,
        use_provenance_multiplier=False,
        lambda_repair=1.5,
        dual_multiplier=1.5,
    )
)

provenance_variant_definition = pd.DataFrame([
    {
        "arm": "revised_P1_provenance_rank_only",
        "provenance_in_ranking": 1,
        "provenance_in_weight_multiplier": 0,
        "selected_slices": p1_selected_slices,
    },
    {
        "arm": "revised_P2_provenance_weight_only",
        "provenance_in_ranking": 0,
        "provenance_in_weight_multiplier": 1,
        "selected_slices": p2_selected_slices,
    },
    {
        "arm": "revised_P3_provenance_both",
        "provenance_in_ranking": 1,
        "provenance_in_weight_multiplier": 1,
        "selected_slices": p3_selected_slices,
    },
    {
        "arm": "revised_P4_no_provenance",
        "provenance_in_ranking": 0,
        "provenance_in_weight_multiplier": 0,
        "selected_slices": p4_selected_slices,
    },
])

display(provenance_variant_definition)

provenance_variant_definition.to_csv(
    "nb9_revised_provenance_variant_definitions.csv",
    index=False
)

print("\nNo-provenance TOP10:")
print(no_provenance_top10)

In [ ]:
# ============================================================
# DOWNSTREAM REPAIR ARMS FOR ASAP GUARDRAIL ABLATIONS
# ============================================================

ablation_top10_long = pd.read_csv(
    "nb8_asap_guardrail_ablation_top10.csv"
)

required_ablation_columns = [
    "configuration",
    "rank",
    "slice_name",
]

missing_ablation_columns = [
    column
    for column in required_ablation_columns
    if column not in ablation_top10_long.columns
]

if missing_ablation_columns:
    raise KeyError(
        "The Notebook 8 guardrail TOP-K file is missing: "
        f"{missing_ablation_columns}"
    )

asap_ablation_weight_columns = {}

for configuration, group in (
    ablation_top10_long
    .groupby("configuration")
):
    group = group.sort_values("rank")

    # Patient weighting must use canonical membership identities.
    selection_column = (
        "membership_name"
        if "membership_name" in group.columns
        else "slice_name"
    )

    selected_slices = (
        group[selection_column]
        .dropna()
        .astype(str)
        .str.strip()
        .drop_duplicates()
        .tolist()
    )

    selected_slices = [
        slice_name
        for slice_name in selected_slices
        if slice_name in slice_membership.columns
    ]

    if not selected_slices:
        raise ValueError(
            f"No reconstructable slices were found for "
            f"configuration: {configuration}"
        )

    safe_name = (
        configuration
        .replace(" ", "_")
        .replace("-", "_")
    )

    weight_col = (
        f"w_asap_ablation__{safe_name}"
    )

    df[weight_col] = (
        build_ranked_sample_weights(
            selected_slices=selected_slices,
            use_provenance_multiplier=False,
            lambda_repair=1.5,
            dual_multiplier=1.5,
        )
    )

    asap_ablation_weight_columns[
        f"asap_ablation__{safe_name}"
    ] = weight_col

print("Created guardrail-ablation weights:")

for arm_name, weight_col in (
    asap_ablation_weight_columns.items()
):
    changed_mask = ~np.isclose(
        df[weight_col].to_numpy(dtype=float),
        1.0
    )

    rows_changed = int(
        changed_mask.sum()
    )

    print(
        arm_name,
        "->",
        weight_col,
        "| rows changed:",
        rows_changed
    )

In [ ]:
# ============================================================
# SELECT REPRESENTATIVE GRID SCHEMES FOR RETRAINING
# Must run before repair_arms are defined and trained.
# ============================================================

grid_summary = pd.read_csv(
    "nb8_asap_constrained_weight_grid_summary.csv"
)

grid_rankings = pd.read_csv(
    "nb8_asap_constrained_weight_grid_rankings.csv"
)

required_grid_summary_columns = [
    "scheme",
    "w_harm",
    "w_provenance",
    "w_support",
    "top10_jaccard",
    "full_ranking_spearman",
]

missing_grid_summary_columns = [
    column
    for column in required_grid_summary_columns
    if column not in grid_summary.columns
]

if missing_grid_summary_columns:
    raise KeyError(
        "Grid summary is missing: "
        f"{missing_grid_summary_columns}"
    )

required_grid_ranking_columns = [
    "scheme",
    "grid_rank",
    "slice_name",
]

missing_grid_ranking_columns = [
    column
    for column in required_grid_ranking_columns
    if column not in grid_rankings.columns
]

if missing_grid_ranking_columns:
    raise KeyError(
        "Grid rankings are missing: "
        f"{missing_grid_ranking_columns}"
    )

# ------------------------------------------------------------
# Identify representative schemes
# ------------------------------------------------------------

distance_to_reference = (
    (grid_summary["w_harm"] - 0.60) ** 2
    +
    (grid_summary["w_provenance"] - 0.30) ** 2
    +
    (grid_summary["w_support"] - 0.10) ** 2
)

reference_scheme = grid_summary.loc[
    distance_to_reference.idxmin(),
    "scheme"
]

harm_heavy_scheme = (
    grid_summary.sort_values(
        ["w_harm", "w_provenance"],
        ascending=[False, True]
    )
    .iloc[0]["scheme"]
)

provenance_heavy_scheme = (
    grid_summary.sort_values(
        ["w_provenance", "w_harm"],
        ascending=[False, True]
    )
    .iloc[0]["scheme"]
)

support_heavy_scheme = (
    grid_summary.sort_values(
        ["w_support", "w_harm"],
        ascending=[False, False]
    )
    .iloc[0]["scheme"]
)

least_stable_scheme = (
    grid_summary.sort_values(
        [
            "top10_jaccard",
            "full_ranking_spearman",
        ],
        ascending=[True, True]
    )
    .iloc[0]["scheme"]
)

representative_grid_schemes = list(
    dict.fromkeys([
        reference_scheme,
        harm_heavy_scheme,
        provenance_heavy_scheme,
        support_heavy_scheme,
        least_stable_scheme,
    ])
)

print(
    "Representative grid schemes:",
    representative_grid_schemes
)

# ------------------------------------------------------------
# Create their row-level repair weights
# ------------------------------------------------------------

grid_weight_columns = {}

for scheme in representative_grid_schemes:
    selected_slices = (
        grid_rankings.loc[
            grid_rankings["scheme"].eq(scheme)
        ]
        .sort_values("grid_rank")
        .head(10)["slice_name"]
        .dropna()
        .astype(str)
        .str.strip()
        .drop_duplicates()
        .tolist()
    )

    selected_slices = [
        slice_name
        for slice_name in selected_slices
        if slice_name in slice_membership.columns
    ]

    if not selected_slices:
        raise ValueError(
            f"No valid TOP10 slices found for {scheme}."
        )

    safe_scheme = (
        scheme
        .replace(".", "p")
        .replace("-", "_")
    )

    weight_col = (
        f"w_grid_repair__{safe_scheme}"
    )

    arm_name = (
        f"grid_repair__{safe_scheme}"
    )

    df[weight_col] = (
        build_ranked_sample_weights(
            selected_slices=selected_slices,
            use_provenance_multiplier=False,
            lambda_repair=1.5,
            dual_multiplier=1.5,
        )
    )

    grid_weight_columns[
        arm_name
    ] = weight_col

print("Created representative grid weights:")

for arm_name, weight_col in (
    grid_weight_columns.items()
):
    print(
        arm_name,
        "->",
        weight_col
    )

# New ranker Ek = H + D + S. End**************************************************

In [ ]:
# ============================================================
# Define modeling matrix safely
# ============================================================

y_col = "y"

base_exclude_cols = {
    y_col,
    "disposition",
    "arrival_hour_bin_raw",

    # OOF diagnostics and labels
    "oof_pred_proba", "oof_logloss", "oof_pred_label_05",
    "oof_error_05", "false_negative_05", "high_conf_contradiction",

    # previous model probabilities
    "p_oof", "p_harm_weighted", "p_harm_weighted_plus_denoise",

    # non-feature diagnostic flags
    "suspicious_targeted_flag",
}

categorical_cols = [
    "arrival_mode",
    "arrival_month",
    "arrival_day",
    "arrival_hour_bin"
]

categorical_cols = [c for c in categorical_cols if c in df.columns]

numeric_cols = [
    c for c in df.columns
    if c not in base_exclude_cols
    and c not in categorical_cols
    and not c.startswith("w_")   # all repair weights must be excluded as features
    and not c.startswith("p_")   # all model probabilities must be excluded as features
    and pd.api.types.is_numeric_dtype(df[c])
]

X_cols = numeric_cols + categorical_cols

X = df[X_cols].copy()
y = df[y_col].astype(int)

print("Numeric cols:", len(numeric_cols))
print("Categorical cols:", categorical_cols)
print("Bad object columns in numeric set:",
      X[numeric_cols].select_dtypes(include=["object", "string"]).columns.tolist())
print("All text columns in X:",
      X.select_dtypes(include=["object", "string"]).columns.tolist())

In [ ]:
print("Still object columns in numeric set:",
      X[numeric_cols].select_dtypes(include=["object", "string"]).columns.tolist())

In [ ]:
## ===== TARGET =====
#y_col = "y"
#
## ===== REMOVE LEAKAGE / NON-FEATURES =====
#drop_cols = [
#    y_col,
#    "oof_pred_proba", "oof_logloss", "oof_pred_label_05",
#    "oof_error_05", "false_negative_05", "high_conf_contradiction",
#    "p_oof"
#]
#
## keep only valid columns
#feature_df = df.drop(columns=[c for c in drop_cols if c in df.columns])
#
## ===== DEFINE TYPES PROPERLY =====
#categorical_cols = [
#    "arrival_mode",
#    "arrival_month",
#    "arrival_day",
#    "arrival_hour_bin"
#]
#
## sanity check — only keep ones that exist
#categorical_cols = [c for c in categorical_cols if c in feature_df.columns]
#
#numeric_cols = [c for c in feature_df.columns if c not in categorical_cols]
#
#print("Numeric cols:", len(numeric_cols))
#print("Categorical cols:", categorical_cols)

In [ ]:
#X = feature_df.copy()
#y = df[y_col].astype(int)

In [ ]:
#y_col = "y"
#
#leakage_or_nonfeature_cols = [
#    y_col,
#    "disposition",
#    "arrival_hour_bin_raw",
#
#    "oof_pred_proba", "oof_logloss", "oof_pred_label_05",
#    "oof_error_05", "false_negative_05", "high_conf_contradiction",
#    "p_oof", "p_harm_weighted", "p_harm_weighted_plus_denoise",
#    "w_final"
#]
#
#categorical_cols = [
#    "arrival_mode",
#    "arrival_month",
#    "arrival_day",
#    "arrival_hour_bin"
#]
#
#categorical_cols = [c for c in categorical_cols if c in df.columns]
#
#numeric_cols = [
#    c for c in df.columns
#    if c not in leakage_or_nonfeature_cols
#    and c not in categorical_cols
#    and pd.api.types.is_numeric_dtype(df[c])
#]
#
#X_cols = numeric_cols + categorical_cols
#
#X = df[X_cols].copy()
#y = df[y_col].astype(int)
#
#print("Numeric cols:", len(numeric_cols))
#print("Categorical cols:", categorical_cols)
#print("Bad object columns in numeric set:",
#      X[numeric_cols].select_dtypes(include=["object", "string"]).columns.tolist())
#print("All text columns in X:",
#      X.select_dtypes(include=["object", "string"]).columns.tolist())

In [ ]:
#print("Still object columns in numeric set:",
#      X[numeric_cols].select_dtypes(include=["object", "string"]).columns.tolist())

Define model training function

In [ ]:
#def train_oof_weighted_model(
#    X,
#    y,
#    sample_weight,
#    numeric_cols,
#    categorical_cols,
#    n_splits=5,
#    random_state=42
#):
#    import numpy as np
#    from sklearn.model_selection import StratifiedKFold
#    from sklearn.compose import ColumnTransformer
#    from sklearn.pipeline import Pipeline
#    from sklearn.linear_model import LogisticRegression
#    from sklearn.impute import SimpleImputer
#    from sklearn.preprocessing import OneHotEncoder, StandardScaler
#
#    preprocess = ColumnTransformer([
#        ("num", Pipeline([
#            ("imputer", SimpleImputer(strategy="median")),
#            ("scaler", StandardScaler())
#        ]), numeric_cols),
#
#        ("cat", Pipeline([
#            ("imputer", SimpleImputer(strategy="most_frequent")),
#            ("onehot", OneHotEncoder(handle_unknown="ignore"))
#        ]), categorical_cols),
#    ])
#
#    model = Pipeline([
#        ("preprocess", preprocess),
#        ("lr", LogisticRegression(max_iter=5000, solver="lbfgs"))
#    ])
#
#    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
#    oof = np.zeros(len(X), dtype=float)
#
#    for fold, (tr, te) in enumerate(cv.split(X, y), 1):
#        model.fit(
#            X.iloc[tr],
#            y.iloc[tr],
#            lr__sample_weight=np.asarray(sample_weight)[tr]
#        )
#        oof[te] = model.predict_proba(X.iloc[te])[:, 1]
#        print(f"Fold {fold} done")
#
#    return np.clip(oof, 1e-6, 1 - 1e-6)

In [ ]:
def train_oof_weighted_model(
    X,
    y,
    sample_weight,
    numeric_cols,
    categorical_cols,
    n_splits=5,
    random_state=42
):
    import numpy as np
    from sklearn.model_selection import StratifiedKFold
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LogisticRegression
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),
    ])

    model = Pipeline([
        ("preprocess", preprocess),
        ("lr", LogisticRegression(max_iter=5000, solver="lbfgs"))
    ])

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof = np.zeros(len(X), dtype=float)

    for fold, (tr, te) in enumerate(cv.split(X, y), 1):
        fit_kwargs = {}

        if sample_weight is not None:
            fit_kwargs["lr__sample_weight"] = np.asarray(sample_weight)[tr]

        model.fit(
            X.iloc[tr],
            y.iloc[tr],
            **fit_kwargs
        )

        oof[te] = model.predict_proba(X.iloc[te])[:, 1]
        print(f"Fold {fold} done")

    return np.clip(oof, 1e-6, 1 - 1e-6)


Train all targeted repair arms

In [ ]:
#epair_arms = {
#   "harm_weighted": "w_final",
#
#
#or arm_name, weight_col in repair_arms.items():
#   print("\nTraining:", arm_name)
#
#   df[f"p_{arm_name}"] = train_oof_weighted_model(
#       X=X,
#       y=y,
#       sample_weight=df[weight_col].values,
#       numeric_cols=numeric_cols,
#       categorical_cols=categorical_cols,
#       n_splits=5,
#       random_state=42
#   )

In [ ]:
## ============================================================
## Train all targeted and source-ablation repair arms
## ============================================================
#
#repair_arms = {
#    # Baseline control
#    "ERM_unweighted": None,
#
#    # Notebook 8 harm-weighted repair
#    "harm_weighted_nb8": "w_final",
#
#    # Notebook 9 targeted harm repair variants
#    "topk_harm": "w_topk",
#    "squared_harm": "w_squared_harm",
#    "worst_slice_boost": "w_worst_boost",
#    "combined_targeted": "w_combined_targeted",
#    "combined_targeted_suspicious": "w_combined_targeted_suspicious",
#
#    # New dual-source discovery ablation arms
#   "clinical_discovery_repair": "w_clinical_discovery",
#    "error_guided_discovery_repair": "w_error_guided_discovery",
#    "dual_supported_repair": "w_dual_supported",
#    "dual_source_combined_discovery_repair": "w_dual_source_combined",
#    "basic_slice_ranked_repair": "w_basic_slice_ranked",
#    "basic_slice_ranked_top5": "w_basic_slice_ranked_top5",
#    "basic_slice_ranked_top7": "w_basic_slice_ranked_top7",
#    "basic_slice_ranked_top10": "w_basic_slice_ranked_top10",
#    "basic_slice_ranked_official_TOP10": "w_basic_slice_ranked_official",
#}
#
## Keep only arms whose weight columns exist.
#available_repair_arms = {}
#
#for arm_name, weight_col in repair_arms.items():
#    if weight_col is None:
#        available_repair_arms[arm_name] = None
#    elif weight_col in df.columns:
#        available_repair_arms[arm_name] = weight_col
#    else:
#        print(f"Skipping {arm_name}: missing weight column {weight_col}")
#
#repair_arms = available_repair_arms
#
#for arm_name, weight_col in repair_arms.items():
#    print("\nTraining:", arm_name)
#
#    sample_weight = None if weight_col is None else df[weight_col].values
#
#    df[f"p_{arm_name}"] = train_oof_weighted_model(
#        X=X,
#        y=y,
#        sample_weight=sample_weight,
#        numeric_cols=numeric_cols,
#        categorical_cols=categorical_cols,
#        n_splits=5,
#        random_state=42
#    )
#

# New ranker EDIT 2 Ek = H + D + S. Start**************************************************

In [ ]:
#required_nb9_objects = [
#    "df",
#    "X",
#    "y",
#    "slice_membership",
#    "repair_arms",
#]
#
#missing_nb9_objects = [
#    object_name
#    for object_name in required_nb9_objects
#    if object_name not in globals()
#]
#
#if missing_nb9_objects:
#    raise NameError(
#        "Notebook 9 is not ready for training. "
#        f"Missing objects: {missing_nb9_objects}"
#    )
#
#required_revised_weight_columns = [
#    "w_revised_P1_provenance_rank_only",
#    "w_revised_P2_provenance_weight_only",
#    "w_revised_P3_provenance_both",
#    "w_revised_P4_no_provenance",
#]
#
#missing_weight_columns = [
#    column
#    for column in required_revised_weight_columns
#    if column not in df.columns
#]
#
#if missing_weight_columns:
#    raise KeyError(
#        "Revised repair weights are missing: "
#        f"{missing_weight_columns}"
#    )
#
#print(
#    "Notebook 9 dependencies are complete."
#)

In [ ]:
provenance_weight_columns = {
    "P1":
        "w_revised_P1_provenance_rank_only",

    "P2":
        "w_revised_P2_provenance_weight_only",

    "P3":
        "w_revised_P3_provenance_both",

    "P4":
        "w_revised_P4_no_provenance",
}

comparison_rows = []

arm_names = list(
    provenance_weight_columns
)

for left_index in range(
    len(arm_names)
):
    for right_index in range(
        left_index + 1,
        len(arm_names)
    ):
        left_arm = arm_names[
            left_index
        ]

        right_arm = arm_names[
            right_index
        ]

        left_values = df[
            provenance_weight_columns[
                left_arm
            ]
        ].to_numpy()

        right_values = df[
            provenance_weight_columns[
                right_arm
            ]
        ].to_numpy()

        absolute_difference = np.abs(
            left_values - right_values
        )

        comparison_rows.append({
            "arm_1": left_arm,
            "arm_2": right_arm,

            "n_rows_different":
                int(
                    (
                        absolute_difference
                        > 1e-12
                    ).sum()
                ),

            "mean_absolute_difference":
                float(
                    absolute_difference.mean()
                ),

            "maximum_absolute_difference":
                float(
                    absolute_difference.max()
                ),
        })

provenance_weight_comparison = (
    pd.DataFrame(
        comparison_rows
    )
)

display(
    provenance_weight_comparison
)

if (
    provenance_weight_comparison[
        "n_rows_different"
    ].max()
    == 0
):
    raise ValueError(
        "All P1-P4 weight vectors are identical. "
        "Do not train until provenance variation "
        "and selected slices are checked."
    )

In [ ]:
# ============================================================
# DEFINE ALL REPAIR ARMS
# ============================================================

repair_arms = {
    # Baseline
    "ERM_unweighted": None,

    # Existing repair comparators
    "harm_weighted_nb8":
        "w_final",

    "topk_harm":
        "w_topk",

    "squared_harm":
        "w_squared_harm",

    "worst_slice_boost":
        "w_worst_boost",

    "combined_targeted":
        "w_combined_targeted",

    "combined_targeted_suspicious":
        "w_combined_targeted_suspicious",

    "clinical_discovery_repair":
        "w_clinical_discovery",

    "error_guided_discovery_repair":
        "w_error_guided_discovery",

    "dual_supported_repair":
        "w_dual_supported",

    "dual_source_combined_discovery_repair":
        "w_dual_source_combined",

    # Legacy seven-criterion BASiC comparators
    "basic_slice_ranked_repair":
        "w_basic_slice_ranked",

    "basic_slice_ranked_top5":
        "w_basic_slice_ranked_top5",

    "basic_slice_ranked_top7":
        "w_basic_slice_ranked_top7",

    # Rename the principal legacy comparator.
    "legacy_BASIC_TOP10":
        "w_basic_slice_ranked_top10",

    # Revised provenance-placement experiment
    "revised_P1_provenance_rank_only":
        "w_revised_P1_provenance_rank_only",

    "revised_P2_provenance_weight_only":
        "w_revised_P2_provenance_weight_only",

    "revised_P3_provenance_both":
        "w_revised_P3_provenance_both",

    "revised_P4_no_provenance":
        "w_revised_P4_no_provenance",
}

# Add governance-ablation arms.
repair_arms.update(
    asap_ablation_weight_columns
)

# Add representative grid arms.
repair_arms.update(
    grid_weight_columns
)

# ------------------------------------------------------------
# Remove only genuinely unavailable optional arms
# ------------------------------------------------------------

available_repair_arms = {}

for arm_name, weight_col in repair_arms.items():
    if weight_col is None:
        available_repair_arms[
            arm_name
        ] = None

    elif weight_col in df.columns:
        available_repair_arms[
            arm_name
        ] = weight_col

    else:
        print(
            f"Skipping optional arm {arm_name}: "
            f"missing {weight_col}"
        )

repair_arms = available_repair_arms

print(
    "Number of repair arms:",
    len(repair_arms)
)

for arm_name, weight_col in repair_arms.items():
    print(
        arm_name,
        "->",
        weight_col
    )

In [ ]:
## ============================================================
## CANONICAL REPORTING ARM SETS
## ============================================================
#
#ARM_ERM = "ERM_unweighted"
#ARM_LEGACY_TOP10 = "legacy_BASIC_TOP10"
#ARM_COMBINED = "combined_targeted"
#
#LEGACY_COMPARATOR_ARMS = [
#    ARM_ERM,
#    "basic_slice_ranked_repair",
#    "basic_slice_ranked_top5",
#    "basic_slice_ranked_top7",
#    ARM_LEGACY_TOP10,
#    ARM_COMBINED,
#]
#
#PROVENANCE_ARMS = [
#    ARM_ERM,
#    ARM_LEGACY_TOP10,
#    "revised_P1_provenance_rank_only",
#    "revised_P2_provenance_weight_only",
#    "revised_P3_provenance_both",
#    "revised_P4_no_provenance",
#    ARM_COMBINED,
#]
#
#def keep_trained_arms(
#    requested_arms,
#    dataframe,
#):
#    retained = [
#        arm_name
#        for arm_name in requested_arms
#        if f"p_{arm_name}" in dataframe.columns
#    ]
#
#    missing = [
#        arm_name
#        for arm_name in requested_arms
#        if f"p_{arm_name}" not in dataframe.columns
#    ]
#
#    if missing:
#        print(
#            "Requested arms without predictions:",
#            missing
#        )
#
#    return retained

In [ ]:
# ============================================================
# CANONICAL REPORTING ARM SETS
# One official proposed configuration; P1/P2/P4 are ablations
# ============================================================

ARM_ERM = "ERM_unweighted"

ARM_LEGACY_TOP10 = (
    "legacy_BASIC_TOP10"
)

OFFICIAL_REVISED_ARM = (
    "revised_P3_provenance_both"
)

ARM_COMBINED_TARGETED = (
    "combined_targeted"
)


PRIMARY_METHOD_ARMS = [
    ARM_ERM,
    ARM_LEGACY_TOP10,
    OFFICIAL_REVISED_ARM,
    ARM_COMBINED_TARGETED,
]


PROVENANCE_ABLATION_ARMS = [
    "revised_P1_provenance_rank_only",
    "revised_P2_provenance_weight_only",
    "revised_P3_provenance_both",
    "revised_P4_no_provenance",
]
LEGACY_COMPARATOR_ARMS = [
    ARM_ERM,
    "basic_slice_ranked_repair",
    "basic_slice_ranked_top5",
    "basic_slice_ranked_top7",
    ARM_LEGACY_TOP10,
    ARM_COMBINED_TARGETED,
]

def keep_trained_arms(
    requested_arms,
    dataframe,
):
    retained = [
        arm_name
        for arm_name in requested_arms
        if f"p_{arm_name}"
        in dataframe.columns
    ]

    missing = [
        arm_name
        for arm_name in requested_arms
        if f"p_{arm_name}"
        not in dataframe.columns
    ]

    if missing:
        print(
            "Requested arms without predictions:",
            missing,
        )

    return retained


print(
    "Official revised BASiC arm:",
    OFFICIAL_REVISED_ARM,
)

In [ ]:
ci_arms = keep_trained_arms(
    LEGACY_COMPARATOR_ARMS,
    df,
)

In [ ]:
arms_to_check = keep_trained_arms(
    LEGACY_COMPARATOR_ARMS
    + [
        "dual_supported_repair",
        "dual_source_combined_discovery_repair",
    ],
    df,
)

In [ ]:
# ============================================================
# PRE-TRAINING DEPENDENCY AND FEATURE VALIDATION
# ============================================================

required_nb9_objects = [
    "df",
    "X",
    "y",
    "slice_membership",
    "numeric_cols",
    "categorical_cols",
    "repair_arms",
    "train_oof_weighted_model",
]

missing_nb9_objects = [
    object_name
    for object_name in required_nb9_objects
    if object_name not in globals()
]

if missing_nb9_objects:
    raise NameError(
        "Notebook 9 is not ready for training. "
        f"Missing objects: {missing_nb9_objects}"
    )

if len(X) != len(df):
    raise ValueError(
        f"X has {len(X)} rows but df has {len(df)} rows."
    )

if len(y) != len(df):
    raise ValueError(
        f"y has {len(y)} rows but df has {len(df)} rows."
    )

if not X.index.equals(df.index):
    raise ValueError(
        "X and df indices are not aligned."
    )

if not slice_membership.index.equals(df.index):
    raise ValueError(
        "slice_membership and df indices are not aligned."
    )

missing_numeric_features = [
    column
    for column in numeric_cols
    if column not in X.columns
]

missing_categorical_features = [
    column
    for column in categorical_cols
    if column not in X.columns
]

if missing_numeric_features:
    raise KeyError(
        "Numeric model features are missing: "
        f"{missing_numeric_features}"
    )

if missing_categorical_features:
    raise KeyError(
        "Categorical model features are missing: "
        f"{missing_categorical_features}"
    )

overlapping_feature_columns = sorted(
    set(numeric_cols)
    &
    set(categorical_cols)
)

if overlapping_feature_columns:
    raise ValueError(
        "Features appear in both numeric and categorical lists: "
        f"{overlapping_feature_columns}"
    )

unrecognised_model_columns = [
    column
    for column in X.columns
    if column not in numeric_cols
    and column not in categorical_cols
]

if unrecognised_model_columns:
    print(
        "Warning: X contains columns not used by the model:",
        unrecognised_model_columns
    )

required_primary_arms = [
    "ERM_unweighted",
    "legacy_BASIC_TOP10",
    "revised_P1_provenance_rank_only",
    "revised_P2_provenance_weight_only",
    "revised_P3_provenance_both",
    "revised_P4_no_provenance",
]

missing_primary_arms = [
    arm_name
    for arm_name in required_primary_arms
    if arm_name not in repair_arms
]

if missing_primary_arms:
    raise KeyError(
        "Primary experiment arms are missing: "
        f"{missing_primary_arms}"
    )

invalid_weight_columns = []

for arm_name, weight_col in repair_arms.items():
    if weight_col is None:
        continue

    if weight_col not in df.columns:
        invalid_weight_columns.append({
            "arm": arm_name,
            "problem": "missing column",
            "weight_col": weight_col,
        })
        continue

    weight_values = pd.to_numeric(
        df[weight_col],
        errors="coerce"
    )

    if weight_values.isna().any():
        invalid_weight_columns.append({
            "arm": arm_name,
            "problem": "contains NaN/non-numeric values",
            "weight_col": weight_col,
        })

    if not np.isfinite(
        weight_values.to_numpy()
    ).all():
        invalid_weight_columns.append({
            "arm": arm_name,
            "problem": "contains non-finite values",
            "weight_col": weight_col,
        })

    if (weight_values <= 0).any():
        invalid_weight_columns.append({
            "arm": arm_name,
            "problem": "contains non-positive values",
            "weight_col": weight_col,
        })

if invalid_weight_columns:
    display(
        pd.DataFrame(
            invalid_weight_columns
        )
    )

    raise ValueError(
        "One or more repair-weight columns are invalid."
    )

print(
    "Notebook 9 pre-training validation passed."
)

In [ ]:
required_nonidentical_pairs = {
    ("P1", "P2"),
    ("P1", "P3"),
    ("P2", "P4"),
    ("P3", "P4"),
}

observed_nonidentical_pairs = set(
    provenance_weight_comparison.loc[
        provenance_weight_comparison[
            "n_rows_different"
        ].gt(0),
        ["arm_1", "arm_2"]
    ]
    .itertuples(
        index=False,
        name=None
    )
)

if not (
    required_nonidentical_pairs
    &
    observed_nonidentical_pairs
):
    raise ValueError(
        "The provenance-placement experiment does not "
        "produce sufficiently distinct weight vectors."
    )

print(
    "Provenance-placement weights are identifiable."
)

In [ ]:
t_runtime_repair = start_timer(
    "Full repair-arm five-fold OOF retraining"
)
# ============================================================
# TRAIN ALL ARMS USING THE SAME FIVE-FOLD OOF FRAMEWORK
# ============================================================

prediction_columns = {}

for arm_name, weight_col in repair_arms.items():
    print("\nTraining:", arm_name)

    sample_weight = (
        None
        if weight_col is None
        else df[
            weight_col
        ].to_numpy(dtype=float)
    )

    probabilities = train_oof_weighted_model(
        X=X,
        y=y,
        sample_weight=sample_weight,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        n_splits=5,
        random_state=42,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float
    )

    if len(probabilities) != len(df):
        raise ValueError(
            f"{arm_name} returned "
            f"{len(probabilities)} predictions; "
            f"expected {len(df)}."
        )

    if not np.isfinite(
        probabilities
    ).all():
        raise ValueError(
            f"{arm_name} produced non-finite predictions."
        )

    if (
        (probabilities < 0).any()
        or
        (probabilities > 1).any()
    ):
        raise ValueError(
            f"{arm_name} produced probabilities "
            "outside [0, 1]."
        )

    prediction_columns[
        f"p_{arm_name}"
    ] = probabilities

prediction_df = pd.DataFrame(
    prediction_columns,
    index=df.index
)

old_probability_columns = [
    column
    for column in prediction_df.columns
    if column in df.columns
]

if old_probability_columns:
    df = df.drop(
        columns=old_probability_columns
    )

df = pd.concat(
    [
        df,
        prediction_df,
    ],
    axis=1
)

print(
    "\nCreated prediction columns:",
    len(prediction_df.columns)
)

for column in prediction_df.columns:
    print("-", column)
stop_timer(
    "Full repair-arm five-fold OOF retraining",
    t_runtime_repair
)

In [ ]:
# ============================================================
# BENCHMARK OFFICIAL BASiC P3 ARM ONLY
# Does NOT overwrite the main prediction table
# ============================================================

official_p3_weight_col = (
    "w_revised_P3_provenance_both"
)

if official_p3_weight_col not in df.columns:
    raise KeyError(
        f"Missing official P3 weight column: "
        f"{official_p3_weight_col}"
    )

t_runtime_p3 = start_timer(
    "Official BASiC P3 five-fold OOF retraining"
)

_ = train_oof_weighted_model(
    X=X,
    y=y,
    sample_weight=df[
        official_p3_weight_col
    ].to_numpy(dtype=float),
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    n_splits=5,
    random_state=42,
)

stop_timer(
    "Official BASiC P3 five-fold OOF retraining",
    t_runtime_p3
)

In [ ]:
## ============================================================
## TRAIN LEGACY, REVISED, PROVENANCE, AND GUARDRAIL ARMS
## ============================================================
#
#repair_arms = {
#    # --------------------------------------------------------
#    # Baseline
#    # --------------------------------------------------------
#    "ERM_unweighted": None,
#
#    # --------------------------------------------------------
#    # Existing/legacy comparators
#    # --------------------------------------------------------
#    "harm_weighted_nb8": "w_final",
#    "topk_harm": "w_topk",
#    "squared_harm": "w_squared_harm",
#    "worst_slice_boost": "w_worst_boost",
#    "combined_targeted": "w_combined_targeted",
#    "combined_targeted_suspicious":
#        "w_combined_targeted_suspicious",
#
#    "clinical_discovery_repair":
#        "w_clinical_discovery",
#    "error_guided_discovery_repair":
#        "w_error_guided_discovery",
#    "dual_supported_repair":
#        "w_dual_supported",
#    "dual_source_combined_discovery_repair":
#        "w_dual_source_combined",
#
#    # Legacy seven-criterion BASiC comparators
#    "basic_slice_ranked_repair":
#    "w_basic_slice_ranked",
#
#    "basic_slice_ranked_top5":
#        "w_basic_slice_ranked_top5",
#    
#    "basic_slice_ranked_top7":
#        "w_basic_slice_ranked_top7",
#    
#    "basic_slice_ranked_top10":
#        "w_basic_slice_ranked_top10",
#
#    # --------------------------------------------------------
#    # Revised provenance-placement experiment
#    # --------------------------------------------------------
#    "revised_P1_provenance_rank_only":
#        "w_revised_P1_provenance_rank_only",
#
#    "revised_P2_provenance_weight_only":
#        "w_revised_P2_provenance_weight_only",
#
#    "revised_P3_provenance_both":
#        "w_revised_P3_provenance_both",
#
#    "revised_P4_no_provenance":
#        "w_revised_P4_no_provenance",
#}
#
## Add guardrail-ablation arms.
#repair_arms.update(
#    asap_ablation_weight_columns
#)
#
## Keep only valid columns.
#available_repair_arms = {}
#
#for arm_name, weight_col in repair_arms.items():
#    if weight_col is None:
#        available_repair_arms[arm_name] = None
#
#    elif weight_col in df.columns:
#        available_repair_arms[
#            arm_name
#        ] = weight_col
#
#    else:
#        print(
#            f"Skipping {arm_name}: "
#            f"missing weight column {weight_col}"
#        )
#
#repair_arms = available_repair_arms
#
## ------------------------------------------------------------
## Train all arms with the same five-fold OOF framework
## ------------------------------------------------------------
#
#prediction_columns = {}
#
#for arm_name, weight_col in repair_arms.items():
#    print("\nTraining:", arm_name)
#
#    sample_weight = (
#        None
#        if weight_col is None
#        else df[weight_col].to_numpy(dtype=float)
#    )
#
#    prediction_columns[f"p_{arm_name}"] = (
#        train_oof_weighted_model(
#            X=X,
#            y=y,
#            sample_weight=sample_weight,
#            numeric_cols=numeric_cols,
#            categorical_cols=categorical_cols,
#            n_splits=5,
#            random_state=42,
#        )
#    )
#
#prediction_df = pd.DataFrame(
#    prediction_columns,
#    index=df.index
#)
#
## Remove old versions when rerunning
#old_probability_columns = [
#    column
#    for column in prediction_df.columns
#    if column in df.columns
#]
#
#if old_probability_columns:
#    df = df.drop(columns=old_probability_columns)
#
#df = pd.concat(
#    [df.copy(), prediction_df],
#    axis=1
#)
#
#print("\nCreated probability columns:")
#for column in prediction_df.columns:
#    print("-", column)

In [ ]:
# ============================================================
# POST-TRAINING PREDICTION VALIDATION
# ============================================================

required_prediction_arms = [
    "ERM_unweighted",
    "legacy_BASIC_TOP10",
    "revised_P1_provenance_rank_only",
    "revised_P2_provenance_weight_only",
    "revised_P3_provenance_both",
    "revised_P4_no_provenance",
]

required_prediction_arms += list(
    asap_ablation_weight_columns.keys()
)

required_prediction_arms += list(
    grid_weight_columns.keys()
)

missing_prediction_columns = [
    f"p_{arm_name}"
    for arm_name in required_prediction_arms
    if f"p_{arm_name}" not in df.columns
]

if missing_prediction_columns:
    raise KeyError(
        "Training did not create the required predictions: "
        f"{missing_prediction_columns}"
    )

grid_probability_columns = [
    f"p_{arm_name}"
    for arm_name in grid_weight_columns
    if f"p_{arm_name}" in df.columns
]

print(
    "Representative grid prediction columns:",
    len(grid_probability_columns)
)

print(
    "All primary, ablation, and grid predictions exist."
)

In [ ]:
#PRIMARY_REPORTING_ARMS = [
#    "ERM_unweighted",
#    "legacy_BASIC_TOP10",
#    "revised_P1_provenance_rank_only",
#    "revised_P2_provenance_weight_only",
#    "revised_P3_provenance_both",
#    "revised_P4_no_provenance",
#    "combined_targeted",
#]
#
#GRID_REPORTING_ARMS = list(
#    grid_weight_columns.keys()
#)
#
#GUARDRAIL_REPORTING_ARMS = list(
#    asap_ablation_weight_columns.keys()
#)

In [ ]:
PRIMARY_REPORTING_ARMS = (
    PRIMARY_METHOD_ARMS.copy()
)

PROVENANCE_REPORTING_ARMS = (
    PROVENANCE_ABLATION_ARMS.copy()
)

GRID_REPORTING_ARMS = list(
    grid_weight_columns.keys()
)

GUARDRAIL_REPORTING_ARMS = list(
    asap_ablation_weight_columns.keys()
)

In [ ]:
## ============================================================
## SELECT REPRESENTATIVE GRID SCHEMES FOR RETRAINING
## ============================================================
#
#grid_summary = pd.read_csv(
#    "nb8_asap_constrained_weight_grid_summary.csv"
#)
#
#grid_rankings = pd.read_csv(
#    "nb8_asap_constrained_weight_grid_rankings.csv"
#)
#
## Reference 0.60/0.30/0.10 scheme.
#distance_to_reference = (
#    (grid_summary["w_harm"] - 0.60) ** 2
#    +
#    (grid_summary["w_provenance"] - 0.30) ** 2
#    +
#    (grid_summary["w_support"] - 0.10) ** 2
#)
#
#reference_index = distance_to_reference.idxmin()
#
#reference_scheme = grid_summary.loc[
#    reference_index,
#    "scheme"
#]
#
## Select extreme but valid schemes.
#harm_heavy_scheme = (
#    grid_summary.sort_values(
#        ["w_harm", "w_provenance"],
#        ascending=[False, True]
#    ).iloc[0]["scheme"]
#)
#
#provenance_heavy_scheme = (
#    grid_summary.sort_values(
#        ["w_provenance", "w_harm"],
#        ascending=[False, True]
#    ).iloc[0]["scheme"]
#)
#
#support_heavy_scheme = (
#    grid_summary.sort_values(
#        ["w_support", "w_harm"],
#        ascending=[False, False]
#    ).iloc[0]["scheme"]
#)
#
## Least stable valid scheme.
#least_stable_scheme = (
#    grid_summary.sort_values(
#        [
#            "top10_jaccard",
#            "full_ranking_spearman"
#        ],
#        ascending=[True, True]
#    ).iloc[0]["scheme"]
#)
#
#representative_grid_schemes = list(dict.fromkeys([
#    reference_scheme,
#    harm_heavy_scheme,
#    provenance_heavy_scheme,
#    support_heavy_scheme,
#    least_stable_scheme,
#]))
#
#print(
#    "Representative schemes:",
#    representative_grid_schemes
#)
#
#grid_weight_columns = {}
#grid_weight_arrays = {}
#
#for scheme in representative_grid_schemes:
#    selected = (
#        grid_rankings.loc[
#            grid_rankings["scheme"].eq(scheme)
#        ]
#        .sort_values("grid_rank")
#        .head(10)["slice_name"]
#        .dropna()
#        .astype(str)
#        .tolist()
#    )
#
#    safe_scheme = (
#        scheme
#        .replace(".", "p")
#        .replace("-", "_")
#    )
#
#    weight_col = (
#        f"w_grid_repair__{safe_scheme}"
#    )
#
#    grid_weight_arrays[weight_col] = (
#        build_ranked_sample_weights(
#            selected_slices=selected,
#            use_provenance_multiplier=False,
#            lambda_repair=1.5,
#        )
#    )
#
#    grid_weight_columns[
#        f"grid_repair__{safe_scheme}"
#    ] = weight_col
#
## Add all representative grid weights at once
#grid_weight_df = pd.DataFrame(
#    grid_weight_arrays,
#    index=df.index
#)
#
#old_grid_columns = [
#    column
#    for column in grid_weight_df.columns
#    if column in df.columns
#]
#
#if old_grid_columns:
#    df = df.drop(
#        columns=old_grid_columns
#    )
#
#df = pd.concat(
#    [
#        df.copy(),
#        grid_weight_df,
#    ],
#    axis=1
#)
#
#print(
#    "Grid weight columns added:",
#    grid_weight_df.columns.tolist()
#)

In [ ]:
## ============================================================
## Reconstruct the full-ASAP governed slice pool
## Must run before the revised repair-evaluation cell
## ============================================================
#
#import numpy as np
#import pandas as pd
#
## Use whichever registry variable exists in this notebook
#if "basic_registry" in globals():
#    asap_pool = basic_registry.copy()
#elif "registry" in globals():
#    asap_pool = registry.copy()
#else:
#    raise NameError(
#        "Neither 'basic_registry' nor 'registry' exists. "
#        "Run the registry-loading cell before this cell."
#    )
#
## Standardize registry column names
#asap_pool.columns = (
#    asap_pool.columns
#    .astype(str)
#    .str.strip()
#)
#
#if "slice_name" not in asap_pool.columns:
#    raise KeyError(
#        "The registry does not contain 'slice_name'. "
#        f"Available columns are: {asap_pool.columns.tolist()}"
#    )
#
#asap_pool["slice_name"] = (
#    asap_pool["slice_name"]
#    .astype("string")
#    .str.strip()
#)
#
## ------------------------------------------------------------
## Recover the finalized full-ASAP eligibility indicator
## ------------------------------------------------------------
#if "asap_eligible_full" in asap_pool.columns:
#    # Normalize an existing eligibility column to binary 0/1
#    eligibility_text = (
#        asap_pool["asap_eligible_full"]
#        .astype("string")
#        .str.strip()
#        .str.lower()
#    )
#
#    asap_pool["asap_eligible_full"] = (
#        eligibility_text.isin(
#            ["1", "true", "yes", "y", "accepted", "eligible", "pass"]
#        )
#    ).astype(int)
#
#elif "final_accept" in asap_pool.columns:
#    # final_accept is the finalized ASAP governance decision
#    final_accept_text = (
#        asap_pool["final_accept"]
#        .astype("string")
#        .str.strip()
#        .str.lower()
#    )
#
#    asap_pool["asap_eligible_full"] = (
#        final_accept_text.isin(
#            ["1", "true", "yes", "y", "accepted", "eligible", "pass"]
#        )
#    ).astype(int)
#
#else:
#    raise KeyError(
#        "Cannot construct full-ASAP eligibility. The registry contains neither "
#        "'asap_eligible_full' nor 'final_accept'. "
#        f"Available columns are: {asap_pool.columns.tolist()}"
#    )
#
## Remove blank and duplicate slice names
#asap_pool = (
#    asap_pool
#    .dropna(subset=["slice_name"])
#    .loc[lambda d: d["slice_name"].ne("")]
#    .drop_duplicates(subset=["slice_name"], keep="last")
#    .reset_index(drop=True)
#)
#
#print("ASAP pool shape:", asap_pool.shape)
#print(
#    "Full-ASAP eligible slices:",
#    int(asap_pool["asap_eligible_full"].sum())
#)
#
#display(
#    asap_pool[
#        ["slice_name", "asap_eligible_full"]
#    ].sort_values(
#        ["asap_eligible_full", "slice_name"],
#        ascending=[False, True]
#    )
#)

In [ ]:
# ============================================================
# LOAD FINAL NOTEBOOK 8 ASAP AUDIT
# ============================================================

asap_pool = pd.read_csv(
    "nb8_asap_complete_candidate_audit.csv"
)

asap_pool.columns = (
    asap_pool.columns
    .astype(str)
    .str.strip()
)

required_asap_columns = [
    "slice_name",
    "membership_name",
    "asap_eligible_full",
    "nb8_harm",
]

missing_asap_columns = [
    column
    for column in required_asap_columns
    if column not in asap_pool.columns
]

if missing_asap_columns:
    raise KeyError(
        "Notebook 8 ASAP audit is missing: "
        f"{missing_asap_columns}"
    )

asap_pool["asap_eligible_full"] = (
    pd.to_numeric(
        asap_pool[
            "asap_eligible_full"
        ],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

asap_pool["nb8_harm"] = (
    pd.to_numeric(
        asap_pool["nb8_harm"],
        errors="coerce"
    )
)

print(
    "Loaded Notebook 8 ASAP audit:",
    asap_pool.shape
)

In [ ]:
# ============================================================
# VERIFY THE ASAP POOL LOADED FROM NOTEBOOK 8
# ============================================================

required_asap_columns = [
    "slice_name",
    "asap_eligible_full",
    "nb8_harm",
]

missing_asap_columns = [
    column
    for column in required_asap_columns
    if column not in asap_pool.columns
]

if missing_asap_columns:
    raise KeyError(
        "asap_pool is missing required columns: "
        f"{missing_asap_columns}"
    )

official_repair_slices = (
    asap_pool.loc[
        asap_pool["asap_eligible_full"].eq(1)
        &
        asap_pool["nb8_harm"].gt(0),
        "slice_name"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

missing_membership_slices = [
    slice_name
    for slice_name in official_repair_slices
    if slice_name not in slice_membership.columns
]

official_repair_slices = [
    slice_name
    for slice_name in official_repair_slices
    if slice_name in slice_membership.columns
]

print(
    "Positive-harm full-ASAP repair slices:",
    len(official_repair_slices)
)

print(
    "Missing membership:",
    missing_membership_slices
)

for slice_name in official_repair_slices:
    print("-", slice_name)

if not official_repair_slices:
    raise ValueError(
        "No positive-harm full-ASAP slices are "
        "available for repair evaluation."
    )

print("\nOfficial repair-evaluation slices:")
for slice_name in official_repair_slices:
    print("-", slice_name)

In [ ]:
# ============================================================
# UNIFIED DOWNSTREAM EVALUATION
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    brier_score_loss,
)

def safe_global_metrics(
    y_true,
    y_prob,
    threshold=0.5
):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    return {
        "AUROC": roc_auc_score(
            y_true,
            y_prob
        ),
        "AUPRC": average_precision_score(
            y_true,
            y_prob
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Brier": brier_score_loss(
            y_true,
            y_prob
        ),
    }

revised_global_rows = []
revised_slice_result_frames = []

if (
    "official_repair_slices" not in globals()
    or len(official_repair_slices) == 0
):
    raise RuntimeError(
        "Run the VERIFY THE ASAP POOL LOADED FROM "
        "NOTEBOOK 8 cell before this evaluation."
    )

for arm_name in repair_arms:
    probability_col = f"p_{arm_name}"

    if probability_col not in df.columns:
        continue

    probabilities = df[
        probability_col
    ].values

    global_metrics = safe_global_metrics(
        y,
        probabilities,
        threshold=0.5
    )

    revised_global_rows.append({
        "arm": arm_name,
        **global_metrics,
    })

    per_slice_rows = []

    for slice_name in official_repair_slices:
        mask = (
            slice_membership[slice_name]
            .astype(bool)
            .values
        )

        n_slice = int(mask.sum())

        if n_slice < 50:
            continue

        if np.unique(
            np.asarray(y)[mask]
        ).size < 2:
            continue

        slice_metrics = safe_global_metrics(
            np.asarray(y)[mask],
            probabilities[mask],
            threshold=0.5
        )

        per_slice_rows.append({
            "arm": arm_name,
            "slice_name": slice_name,
            "n": n_slice,
            **slice_metrics,
        })

    if per_slice_rows:
        revised_slice_result_frames.append(
            pd.DataFrame(per_slice_rows)
        )

revised_global_results = pd.DataFrame(
    revised_global_rows
)

if not revised_slice_result_frames:
    raise ValueError(
        "No per-slice evaluation results were produced. "
        "Check official_repair_slices and the prediction columns."
    )

revised_slice_results = pd.concat(
    revised_slice_result_frames,
    ignore_index=True
)

# ------------------------------------------------------------
# Summarize repair-slice robustness
# ------------------------------------------------------------

revised_repair_robustness_rows = []

for arm_name, group in (
    revised_slice_results.groupby("arm")
):
    revised_repair_robustness_rows.append({
        "arm": arm_name,

        "n_evaluated_repair_slices":
            len(group),

        "mean_repair_slice_AUPRC":
            group["AUPRC"].mean(),

        "worst_repair_slice_AUPRC":
            group["AUPRC"].min(),

        "mean_to_worst_AUPRC_gap":
            (
                group["AUPRC"].mean()
                - group["AUPRC"].min()
            ),

        "mean_repair_slice_Recall":
            group["Recall"].mean(),

        "worst_repair_slice_Recall":
            group["Recall"].min(),

        "mean_repair_slice_Brier":
            group["Brier"].mean(),

        "worst_repair_slice_Brier":
            group["Brier"].max(),

        "worst_AUPRC_slice":
            group.loc[
                group["AUPRC"].idxmin(),
                "slice_name"
            ],
    })

revised_repair_robustness = pd.DataFrame(
    revised_repair_robustness_rows
)

display(
    revised_global_results.sort_values(
        "AUPRC",
        ascending=False
    )
)

display(
    revised_repair_robustness.sort_values(
        "worst_repair_slice_AUPRC",
        ascending=False
    )
)

revised_global_results.to_csv(
    "nb9_revised_global_results.csv",
    index=False
)

revised_slice_results.to_csv(
    "nb9_revised_repair_slice_results.csv",
    index=False
)

revised_repair_robustness.to_csv(
    "nb9_revised_repair_robustness.csv",
    index=False
)

In [ ]:
t_runtime_threshold = start_timer(
    "Recall-preserving threshold recalibration"
)
# ============================================================
# REVISED RECALL-PRESERVING THRESHOLD RECALIBRATION
# ============================================================

main_slice = (
    "abdominal_pain_and_hr_missing"
)

if main_slice not in slice_membership.columns:
    raise KeyError(
        f"{main_slice} not in slice_membership."
    )

main_mask = (
    slice_membership[main_slice]
    .astype(bool)
    .values
)

threshold_grid = np.arange(
    0.05,
    0.951,
    0.01
)

erm_probabilities = (
    df["p_ERM_unweighted"].values
)

erm_predictions_05 = (
    erm_probabilities >= 0.50
).astype(int)

target_recall = recall_score(
    np.asarray(y)[main_mask],
    erm_predictions_05[main_mask],
    zero_division=0
)

priority_threshold_arms = [
    "ERM_unweighted",
    "legacy_BASIC_TOP10",
    "revised_P1_provenance_rank_only",
    "revised_P2_provenance_weight_only",
    "revised_P3_provenance_both",
    "revised_P4_no_provenance",
    "combined_targeted",
]

priority_threshold_arms += list(
    asap_ablation_weight_columns.keys()
)
priority_threshold_arms += list(
    grid_weight_columns.keys()
)

priority_threshold_arms = list(
    dict.fromkeys(
        priority_threshold_arms
    )
)
threshold_rows = []

for arm_name in priority_threshold_arms:
    probability_col = f"p_{arm_name}"

    if probability_col not in df.columns:
        continue

    probabilities = df[
        probability_col
    ].values

    candidates = []

    for threshold in threshold_grid:
        predictions = (
            probabilities >= threshold
        ).astype(int)

        recall_value = recall_score(
            np.asarray(y)[main_mask],
            predictions[main_mask],
            zero_division=0
        )

        precision_value = precision_score(
            np.asarray(y)[main_mask],
            predictions[main_mask],
            zero_division=0
        )

        candidates.append({
            "threshold": threshold,
            "Recall": recall_value,
            "Precision": precision_value,
        })

    candidate_table = pd.DataFrame(
        candidates
    )

    feasible = candidate_table[
        candidate_table["Recall"]
        >= target_recall
    ].copy()

    if len(feasible):
        best = feasible.sort_values(
            ["Precision", "threshold"],
            ascending=[False, False]
        ).iloc[0]
    else:
        best = candidate_table.sort_values(
            ["Recall", "Precision"],
            ascending=[False, False]
        ).iloc[0]

    threshold_rows.append({
        "arm": arm_name,
        "slice_name": main_slice,
        "target_ERM_recall_at_0_5":
            target_recall,
        "selected_threshold":
            best["threshold"],
        "recalibrated_Recall":
            best["Recall"],
        "recalibrated_Precision":
            best["Precision"],
        "AUPRC": average_precision_score(
            np.asarray(y)[main_mask],
            probabilities[main_mask]
        ),
        "Brier": brier_score_loss(
            np.asarray(y)[main_mask],
            probabilities[main_mask]
        ),
    })

revised_threshold_recalibration = pd.DataFrame(
    threshold_rows
)

display(
    revised_threshold_recalibration.sort_values(
        ["AUPRC", "Brier"],
        ascending=[False, True]
    )
)

revised_threshold_recalibration.to_csv(
    "nb9_revised_threshold_recalibration.csv",
    index=False
)
stop_timer(
    "Recall-preserving threshold recalibration",
    t_runtime_threshold
)

In [ ]:
t_runtime_bootstrap = start_timer(
    "Principal-slice bootstrap analysis - 1000 replicates"
)
# ============================================================
# REVISED BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

from sklearn.utils import resample

N_BOOT = 1000
BOOTSTRAP_RANDOM_STATE = 42

rng = np.random.default_rng(
    BOOTSTRAP_RANDOM_STATE
)

bootstrap_arms = [
    "ERM_unweighted",
    "legacy_BASIC_TOP10",
    "revised_P1_provenance_rank_only",
    "revised_P2_provenance_weight_only",
    "revised_P3_provenance_both",
    "revised_P4_no_provenance",
    "combined_targeted",
]
bootstrap_arms += list(
    grid_weight_columns.keys()
)

bootstrap_arms = list(
    dict.fromkeys(
        bootstrap_arms
    )
)
def bootstrap_slice_metrics(
    y_true,
    y_prob,
    n_boot=1000,
    threshold=0.5,
    random_state=42,
):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    local_rng = np.random.default_rng(
        random_state
    )

    records = []

    n = len(y_true)

    for bootstrap_id in range(n_boot):
        index = local_rng.integers(
            0,
            n,
            size=n
        )

        y_b = y_true[index]
        p_b = y_prob[index]

        if np.unique(y_b).size < 2:
            continue

        pred_b = (
            p_b >= threshold
        ).astype(int)

        records.append({
            "AUPRC":
                average_precision_score(
                    y_b,
                    p_b
                ),
            "Recall":
                recall_score(
                    y_b,
                    pred_b,
                    zero_division=0
                ),
            "Brier":
                brier_score_loss(
                    y_b,
                    p_b
                ),
        })

    boot = pd.DataFrame(records)

    output = {}

    for metric in [
        "AUPRC",
        "Recall",
        "Brier",
    ]:
        output[f"{metric}_mean"] = (
            boot[metric].mean()
        )

        output[f"{metric}_ci_low"] = (
            boot[metric].quantile(0.025)
        )

        output[f"{metric}_ci_high"] = (
            boot[metric].quantile(0.975)
        )

    output["n_successful_bootstraps"] = (
        len(boot)
    )

    return output

bootstrap_rows = []

for arm_name in bootstrap_arms:
    probability_col = f"p_{arm_name}"

    if probability_col not in df.columns:
        continue

    probabilities = df[
        probability_col
    ].values

    metrics = bootstrap_slice_metrics(
        y_true=np.asarray(y)[main_mask],
        y_prob=probabilities[main_mask],
        n_boot=N_BOOT,
        threshold=0.50,
        random_state=BOOTSTRAP_RANDOM_STATE,
    )

    bootstrap_rows.append({
        "arm": arm_name,
        "slice_name": main_slice,
        **metrics,
    })

revised_bootstrap_main_slice = pd.DataFrame(
    bootstrap_rows
)

display(revised_bootstrap_main_slice)

revised_bootstrap_main_slice.to_csv(
    "nb9_revised_bootstrap_main_slice.csv",
    index=False
)

In [ ]:
# ============================================================
# PAIRED BOOTSTRAP DIFFERENCES VS ERM
# ============================================================

def paired_bootstrap_difference(
    y_true,
    p_baseline,
    p_repair,
    n_boot=1000,
    random_state=42,
):
    y_true = np.asarray(y_true)
    p_baseline = np.asarray(p_baseline)
    p_repair = np.asarray(p_repair)

    local_rng = np.random.default_rng(
        random_state
    )

    rows = []
    n = len(y_true)

    for bootstrap_id in range(n_boot):
        index = local_rng.integers(
            0,
            n,
            size=n
        )

        y_b = y_true[index]

        if np.unique(y_b).size < 2:
            continue

        p0 = p_baseline[index]
        p1 = p_repair[index]

        rows.append({
            "delta_AUPRC": (
                average_precision_score(
                    y_b,
                    p1
                )
                -
                average_precision_score(
                    y_b,
                    p0
                )
            ),
            "delta_Brier_improvement": (
                brier_score_loss(
                    y_b,
                    p0
                )
                -
                brier_score_loss(
                    y_b,
                    p1
                )
            ),
        })

    result = pd.DataFrame(rows)

    summary = {}

    for metric in [
        "delta_AUPRC",
        "delta_Brier_improvement",
    ]:
        summary[f"{metric}_mean"] = (
            result[metric].mean()
        )

        summary[f"{metric}_ci_low"] = (
            result[metric].quantile(0.025)
        )

        summary[f"{metric}_ci_high"] = (
            result[metric].quantile(0.975)
        )

        summary[f"{metric}_prob_gt_zero"] = (
            result[metric] > 0
        ).mean()

    return summary

paired_rows = []

baseline_main_prob = (
    df["p_ERM_unweighted"]
    .values[main_mask]
)

for arm_name in bootstrap_arms:
    if arm_name == "ERM_unweighted":
        continue

    probability_col = f"p_{arm_name}"

    if probability_col not in df.columns:
        continue

    comparison = paired_bootstrap_difference(
        y_true=np.asarray(y)[main_mask],
        p_baseline=baseline_main_prob,
        p_repair=df[
            probability_col
        ].values[main_mask],
        n_boot=N_BOOT,
        random_state=42,
    )

    paired_rows.append({
        "arm": arm_name,
        "comparison": "versus_ERM",
        **comparison,
    })

paired_bootstrap_results = pd.DataFrame(
    paired_rows
)

display(paired_bootstrap_results)

paired_bootstrap_results.to_csv(
    "nb9_revised_paired_bootstrap_vs_ERM.csv",
    index=False
)
stop_timer(
    "Principal-slice bootstrap analysis - 1000 replicates",
    t_runtime_bootstrap
)

In [ ]:
# ============================================================
# FINAL ASAP-CONSTRAINED BASiC REPORTING TABLE
# ============================================================

#primary_final_arms = [
#    "ERM_unweighted",
#    "legacy_BASIC_TOP10",
#    "revised_P1_provenance_rank_only",
#    "revised_P2_provenance_weight_only",
#    "revised_P3_provenance_both",
#    "revised_P4_no_provenance",
#    "combined_targeted",
#]
primary_final_arms = (
    PRIMARY_METHOD_ARMS.copy()
)

global_selected = revised_global_results[
    revised_global_results["arm"]
    .isin(primary_final_arms)
].copy()

robustness_selected = (
    revised_repair_robustness[
        revised_repair_robustness["arm"]
        .isin(primary_final_arms)
    ].copy()
)

threshold_selected = (
    revised_threshold_recalibration[
        revised_threshold_recalibration["arm"]
        .isin(primary_final_arms)
    ].copy()
)

main_slice_selected = (
    revised_slice_results[
        (
            revised_slice_results["arm"]
            .isin(primary_final_arms)
        )
        &
        (
            revised_slice_results["slice_name"]
            == main_slice
        )
    ].copy()
)

final_asap_constrained_report = (
    global_selected
    .merge(
        robustness_selected,
        on="arm",
        how="left",
        suffixes=(
            "_global",
            "_repair"
        )
    )
    .merge(
        main_slice_selected[
            [
                "arm",
                "AUPRC",
                "Recall",
                "Precision",
                "Brier",
            ]
        ].rename(columns={
            "AUPRC": "main_slice_AUPRC",
            "Recall":
                "main_slice_Recall_at_0_5",
            "Precision":
                "main_slice_Precision_at_0_5",
            "Brier": "main_slice_Brier",
        }),
        on="arm",
        how="left",
    )
    .merge(
        threshold_selected[
            [
                "arm",
                "selected_threshold",
                "recalibrated_Recall",
                "recalibrated_Precision",
            ]
        ],
        on="arm",
        how="left",
    )
)

display(final_asap_constrained_report)

final_asap_constrained_report.to_csv(
    "nb9_final_asap_constrained_basic_report.csv",
    index=False
)

In [ ]:
# ============================================================
# PROVENANCE-PLACEMENT ABLATION TABLE
# P1-P4 are ablations, not four official methods
# ============================================================

provenance_ablation_report = (
    revised_global_results.loc[
        revised_global_results[
            "arm"
        ].isin(
            PROVENANCE_ABLATION_ARMS
        )
    ]
    .merge(
        revised_repair_robustness.loc[
            revised_repair_robustness[
                "arm"
            ].isin(
                PROVENANCE_ABLATION_ARMS
            )
        ],
        on="arm",
        how="left",
        suffixes=(
            "_global",
            "_repair",
        ),
    )
)

display(
    provenance_ablation_report
)

provenance_ablation_report.to_csv(
    "nb9_provenance_placement_ablation_report.csv",
    index=False,
)

In [ ]:
# ============================================================
# REPRESENTATIVE WEIGHT-GRID RETRAINING REPORT
# ============================================================

grid_global_results = (
    revised_global_results.loc[
        revised_global_results["arm"]
        .isin(
            grid_weight_columns.keys()
        )
    ]
    .copy()
)

grid_robustness_results = (
    revised_repair_robustness.loc[
        revised_repair_robustness["arm"]
        .isin(
            grid_weight_columns.keys()
        )
    ]
    .copy()
)

grid_retraining_report = (
    grid_global_results
    .merge(
        grid_robustness_results,
        on="arm",
        how="left",
    )
)

display(
    grid_retraining_report.sort_values(
        [
            "worst_repair_slice_AUPRC",
            "AUPRC",
        ],
        ascending=False
    )
)

grid_retraining_report.to_csv(
    "nb9_representative_grid_retraining_report.csv",
    index=False
)

# New ranker EDIT 2 Ek = H + D + S. End**************************************************

In [ ]:
# ============================================================
# ROW-LEVEL DATA REPAIR LINEAGE
# Raw encounter -> diagnostics -> slice membership ->
# repair weight -> baseline probability -> repaired probability
# ============================================================

import numpy as np
import pandas as pd

OFFICIAL_ARM = "legacy_BASIC_TOP10"
BASELINE_ARM = "ERM_unweighted"
MAIN_SLICE = "abdominal_pain_and_hr_missing"

official_weight_col = "w_basic_slice_ranked_top10"
baseline_prob_col = f"p_{BASELINE_ARM}"
repaired_prob_col = f"p_{OFFICIAL_ARM}"

required = [
    official_weight_col,
    baseline_prob_col,
    repaired_prob_col
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(
        f"Missing columns {missing}. Run the weight construction and "
        "all-model training cells first."
    )

# Attach row identifier without changing the model data
lineage = df.copy()
lineage["row_id"] = np.arange(len(lineage))

# Add slice indicators used for explanation
for s in slice_membership.columns:
    lineage[f"SLICE__{s}"] = slice_membership[s].values

# Explain why each row was upweighted
official_slices = (
    pd.read_csv(
        "nb8_asap_constrained_basic_top10.csv"
    )["slice_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

def active_repair_slices(row):
    active = [
        s for s in official_slices
        if f"SLICE__{s}" in row.index and row[f"SLICE__{s}"] == 1
    ]
    return ", ".join(active) if active else "none"

lineage["active_TOP10_repair_slices"] = lineage.apply(
    active_repair_slices,
    axis=1
)

lineage["baseline_probability"] = lineage[baseline_prob_col]
lineage["repaired_probability"] = lineage[repaired_prob_col]
lineage["probability_change"] = (
    lineage["repaired_probability"]
    - lineage["baseline_probability"]
)

lineage["baseline_prediction_05"] = (
    lineage["baseline_probability"] >= 0.5
).astype(int)

lineage["repaired_prediction_05"] = (
    lineage["repaired_probability"] >= 0.5
).astype(int)

# Select examples:
# 3 from the main harmful slice,
# 3 from other upweighted rows,
# 3 unchanged/control rows.
main_mask = slice_membership[MAIN_SLICE].astype(bool).values
upweighted_mask = lineage[official_weight_col].values > 1.001
control_mask = ~upweighted_mask

example_indices = []

example_indices += lineage.loc[main_mask].head(3).index.tolist()

example_indices += lineage.loc[
    upweighted_mask & ~main_mask
].head(3).index.tolist()

example_indices += lineage.loc[
    control_mask
].head(3).index.tolist()

example_indices = list(dict.fromkeys(example_indices))

candidate_cols = [
    "row_id",
    target_col,
    "age",
    "abdominal_pain",
    "hr_missing",
    "heart_rate_final",
    "ckd",
    "spo2_missing",
    "spo2",
    "respiratory",
    "arrival_mode",
    "active_TOP10_repair_slices",
    official_weight_col,
    "baseline_probability",
    "repaired_probability",
    "probability_change",
    "baseline_prediction_05",
    "repaired_prediction_05",
]

show_cols = [c for c in candidate_cols if c in lineage.columns]

row_level_repair_examples = lineage.loc[
    example_indices,
    show_cols
].copy()

display(row_level_repair_examples)

row_level_repair_examples.to_csv(
    "nb9_row_level_repair_examples.csv",
    index=False
)

print("Saved: nb9_row_level_repair_examples.csv")

In [ ]:
# ============================================================
# SAVE A SCREENSHOT-STYLE ROW-LEVEL TABLE
# ============================================================

import matplotlib.pyplot as plt

plot_df = row_level_repair_examples.copy()

# Shorten long text for the figure
if "active_TOP10_repair_slices" in plot_df.columns:
    plot_df["active_TOP10_repair_slices"] = (
        plot_df["active_TOP10_repair_slices"]
        .astype(str)
        .str.slice(0, 35)
    )

# Keep the figure readable
figure_cols = [
    c for c in [
        "row_id",
        target_col,
        "abdominal_pain",
        "hr_missing",
        "active_TOP10_repair_slices",
        official_weight_col,
        "baseline_probability",
        "repaired_probability",
        "probability_change",
    ]
    if c in plot_df.columns
]

figure_df = plot_df[figure_cols].copy().round(4)

fig, ax = plt.subplots(
    figsize=(max(12, len(figure_cols) * 1.8), 4.5)
)
ax.axis("off")

table = ax.table(
    cellText=figure_df.values,
    colLabels=figure_df.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.5)

plt.title(
    "Row-Level Transition from Raw Encounter to BASiC-Ranked Repair",
    pad=20
)
plt.tight_layout()

plt.savefig(
    "nb9_row_level_repair_examples.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Saved: nb9_row_level_repair_examples.png")

In [ ]:
# ============================================================
# SLICE-LEVEL REPAIR LINEAGE TABLE
# ============================================================

rank_table = pd.read_csv("nb8_basic_slice_rank_table.csv")

official_set = set(topk_ranked_slices[10])

slice_lineage = rank_table.copy()

slice_lineage["selected_in_official_TOP10"] = (
    slice_lineage["slice_name"].isin(official_set)
)

slice_lineage["patients_in_slice"] = slice_lineage["slice_name"].map(
    {
        s: int(slice_membership[s].sum())
        for s in slice_membership.columns
    }
).fillna(0).astype(int)

slice_lineage["pct_dataset"] = (
    100 * slice_lineage["patients_in_slice"] / len(df)
)

lineage_cols = [
    "slice_name",
    "nb8_harm",
    "harm_norm",
    "dual_source_evidence",
    "clinical_plausibility",
    "stability_score",
    "support_score",
    "artifact_risk",
    "redundancy_risk",
    "basic_slice_rank_score",
    "selected_in_official_TOP10",
    "patients_in_slice",
    "pct_dataset",
]

lineage_cols = [
    c for c in lineage_cols
    if c in slice_lineage.columns
]

slice_lineage_report = (
    slice_lineage[lineage_cols]
    .sort_values(
        "basic_slice_rank_score",
        ascending=False
    )
)

display(slice_lineage_report)

slice_lineage_report.to_csv(
    "nb9_slice_level_repair_lineage.csv",
    index=False
)

In [ ]:
# ============================================================
# ACTUAL PATIENT-LEVEL BEFORE/AFTER REPAIR DISPLAY
# Run after the model-training cell in Notebook 9
# ============================================================

import numpy as np
import pandas as pd

baseline_arm = "ERM_unweighted"
repaired_arm = "legacy_BASIC_TOP10"
weight_col = "w_basic_slice_ranked_top10"

baseline_probability_col = f"p_{baseline_arm}"
repaired_probability_col = f"p_{repaired_arm}"

required_columns = [
    weight_col,
    baseline_probability_col,
    repaired_probability_col
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing columns: {missing_columns}. "
        "Run the TOP_K weight construction and model-training cells first."
    )

# Preserve the dataframe index as an interpretable patient-row identifier.
patient_transition = df.copy()
patient_transition["row_id"] = np.arange(len(patient_transition))

# Attach slice membership columns for explanation.
for slice_name in slice_membership.columns:
    patient_transition[f"slice__{slice_name}"] = (
        slice_membership[slice_name].values
    )

official_slices = (
    pd.read_csv(
        "nb8_asap_constrained_basic_top10.csv"
    )["slice_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

def identify_active_repair_slices(row):
    active = []

    for slice_name in official_slices:
        indicator = f"slice__{slice_name}"

        if indicator in row.index and row[indicator] == 1:
            active.append(slice_name)

    return ", ".join(active) if active else "none"

patient_transition["active_repair_slices"] = (
    patient_transition.apply(
        identify_active_repair_slices,
        axis=1
    )
)

patient_transition["baseline_probability"] = (
    patient_transition[baseline_probability_col]
)

patient_transition["repaired_probability"] = (
    patient_transition[repaired_probability_col]
)

patient_transition["probability_change"] = (
    patient_transition["repaired_probability"]
    - patient_transition["baseline_probability"]
)

patient_transition["baseline_prediction_05"] = (
    patient_transition["baseline_probability"] >= 0.50
).astype(int)

patient_transition["repaired_prediction_05"] = (
    patient_transition["repaired_probability"] >= 0.50
).astype(int)

# Identify useful examples.
main_slice = "abdominal_pain_and_hr_missing"

main_slice_mask = (
    slice_membership[main_slice].astype(bool).values
)

upweighted_mask = (
    patient_transition[weight_col].values > 1.001
)

unchanged_weight_mask = ~upweighted_mask

# Select:
# 4 rows from the main harmful slice
# 4 rows from other upweighted slices
# 4 ordinary control rows
selected_indices = []

selected_indices.extend(
    patient_transition.loc[main_slice_mask]
    .head(4)
    .index
    .tolist()
)

selected_indices.extend(
    patient_transition.loc[
        upweighted_mask & ~main_slice_mask
    ]
    .head(4)
    .index
    .tolist()
)

selected_indices.extend(
    patient_transition.loc[
        unchanged_weight_mask
    ]
    .head(4)
    .index
    .tolist()
)

selected_indices = list(dict.fromkeys(selected_indices))

candidate_columns = [
    "row_id",
    target_col,

    # Original patient variables
    "age",
    "arrival_mode",
    "abdominal_pain",
    "heart_rate_final",
    "hr_missing",
    "bp_systolic",
    "bp_diastole",
    "respiratory",
    "spo2",
    "spo2_missing",
    "ckd",
    "dm",
    "htn",

    # Added slice explanation
    "active_repair_slices",

    # Added repair field
    weight_col,

    # Before/after model outputs
    "baseline_probability",
    "repaired_probability",
    "probability_change",
    "baseline_prediction_05",
    "repaired_prediction_05",
]

display_columns = [
    column for column in candidate_columns
    if column in patient_transition.columns
]

actual_before_after_rows = (
    patient_transition.loc[
        selected_indices,
        display_columns
    ]
    .copy()
)

display(actual_before_after_rows)

actual_before_after_rows.to_csv(
    "nb9_actual_patient_before_after_repair.csv",
    index=False
)

print(
    "Saved: nb9_actual_patient_before_after_repair.csv"
)

In [ ]:
# ============================================================
# VERIFY THAT ORIGINAL PATIENT CELLS DID NOT CHANGE
# ============================================================

original_feature_columns = [
    column for column in feature_cols
    if column in df.columns
]

# The same dataframe supplies original features to both models.
# We create two labeled views for demonstration only.
before_view = df[original_feature_columns].copy()
after_view = df[original_feature_columns].copy()

comparison = pd.DataFrame({
    "column": original_feature_columns,
    "number_of_changed_cells": [
        (
            before_view[column]
            .fillna("__MISSING__")
            .astype(str)
            !=
            after_view[column]
            .fillna("__MISSING__")
            .astype(str)
        ).sum()
        for column in original_feature_columns
    ]
})

comparison["unchanged"] = (
    comparison["number_of_changed_cells"] == 0
)

display(comparison)

print(
    "Total changed original feature cells:",
    comparison["number_of_changed_cells"].sum()
)

Global evaluation helper

In [ ]:
def evaluate_global(y, p, threshold=0.5):
    pred = (p >= threshold).astype(int)

    return {
        "AUROC": roc_auc_score(y, p),
        "AUPRC": average_precision_score(y, p),
        "Recall": recall_score(y, pred),
        "Precision": precision_score(y, pred),
        "Brier": brier_score_loss(y, p),
    }

Slice metric helper

In [ ]:
def compute_slice_metrics(df, slice_membership, prob_col, target_col, threshold=0.5, min_n=50):
    rows = []

    for s in slice_membership.columns:
        mask = slice_membership[s].values == 1

        if mask.sum() < min_n:
            continue

        y_slice = df.loc[mask, target_col].values
        p_slice = df.loc[mask, prob_col].values
        pred_slice = (p_slice >= threshold).astype(int)

        rows.append({
            "slice": s,
            "n": int(mask.sum()),
            "AUPRC": average_precision_score(y_slice, p_slice),
            "Recall": recall_score(y_slice, pred_slice),
            "Precision": precision_score(y_slice, pred_slice, zero_division=0),
            "Brier": brier_score_loss(y_slice, p_slice),
        })

    return pd.DataFrame(rows)

Evaluate all arms globally

In [ ]:
global_rows = []

for arm_name in repair_arms.keys():
    prob_col = f"p_{arm_name}"
    m = evaluate_global(y, df[prob_col].values, threshold=0.5)
    m["arm"] = arm_name
    global_rows.append(m)

global_nb9 = pd.DataFrame(global_rows)
global_nb9 = global_nb9[["arm", "AUROC", "AUPRC", "Recall", "Precision", "Brier"]]
global_nb9

Evaluate slice robustness and avg-to-worst gap

In [ ]:
robust_rows = []
slice_results_dict = {}

for arm_name in repair_arms.keys():
    prob_col = f"p_{arm_name}"

    sr = compute_slice_metrics(
        df=df,
        slice_membership=slice_membership,
        prob_col=prob_col,
        target_col=target_col,
        threshold=0.5,
        min_n=50
    )

    slice_results_dict[arm_name] = sr

    robust_rows.append({
        "arm": arm_name,
        "mean_slice_AUPRC": sr["AUPRC"].mean(),
        "worst_slice_AUPRC": sr["AUPRC"].min(),
        "avg_to_worst_AUPRC_gap": sr["AUPRC"].mean() - sr["AUPRC"].min(),

        "mean_slice_recall": sr["Recall"].mean(),
        "worst_slice_recall": sr["Recall"].min(),
        "avg_to_worst_recall_gap": sr["Recall"].mean() - sr["Recall"].min(),

        "worst_AUPRC_slice": sr.loc[sr["AUPRC"].idxmin(), "slice"],
        "worst_recall_slice": sr.loc[sr["Recall"].idxmin(), "slice"],
    })

robust_nb9 = pd.DataFrame(robust_rows)
robust_nb9

In [ ]:
## ============================================================
## REPAIR-ELIGIBLE SLICE ROBUSTNESS ONLY
## Excludes monitoring-only / artifact slices such as arrival_mode_unknown
## ============================================================
#########################
#from sklearn.metrics import (
#    recall_score,
#    precision_score,
#    average_precision_score
#)
##########################
#repair_eligible_slices = pd.read_csv(
#    "nb8_repair_eligible_combined_slices.csv"
#)["slice_name"].dropna().astype(str).tolist()
#
#print("Repair-eligible slices:", len(repair_eligible_slices))
#print(repair_eligible_slices)
#
#repair_robust_rows = []
#####################################################
#threshold = 0.5
#
#slice_eval_rows = []
#
#for arm_name in repair_arms.keys():
#
#    p = df[f"p_{arm_name}"].values
#    pred = (p >= threshold).astype(int)
#
#    slice_rows = []
#
#    for s in repair_eligible_slices:
#
#        mask = slice_membership[s] == 1
#
#        if mask.sum() < 30:
#            continue
#
#        y_true = y[mask]
#        y_pred = pred[mask]
#        y_prob = p[mask]
#
#        row = {
#            "arm": arm_name,
#            "slice": s,
#            "n": int(mask.sum()),
#            "recall": recall_score(y_true, y_pred, zero_division=0),
#            "precision": precision_score(y_true, y_pred, zero_division=0),
#            "auprc": average_precision_score(y_true, y_prob)
#        }
#
#        slice_rows.append(row)
#
#    slice_eval_rows.extend(slice_rows)
#
#slice_eval_df = pd.DataFrame(slice_eval_rows)
#
#slice_eval_df.head()
#####################################################
#for arm_name in repair_arms.keys():
#    p = df[f"p_{arm_name}"].values
#    pred = (p >= threshold).astype(int)
#
#    slice_rows = []
#
#    for s in repair_eligible_slices:
#        if s not in slice_membership.columns:
#            continue
#
#        mask = slice_membership[s].astype(bool).values
#
#        if mask.sum() < 30:
#            continue
#
#        if len(np.unique(y[mask])) < 2:
#            continue
#
#        slice_rows.append({
#            "arm": arm_name,
#            "slice_name": s,
#            "support": int(mask.sum()),
#            "slice_AUPRC": average_precision_score(y[mask], p[mask]),
#            "slice_recall": recall_score(y[mask], pred[mask], zero_division=0)
#        })
#
#    temp = pd.DataFrame(slice_rows)
#
#    repair_robust_rows.append({
#        "arm": arm_name,
#        "mean_repair_slice_AUPRC": temp["slice_AUPRC"].mean(),
#        "worst_repair_slice_AUPRC": temp["slice_AUPRC"].min(),
#        "avg_to_worst_repair_AUPRC_gap": temp["slice_AUPRC"].mean() - temp["slice_AUPRC"].min(),
#        "mean_repair_slice_recall": temp["slice_recall"].mean(),
#        "worst_repair_slice_recall": temp["slice_recall"].min(),
#        "worst_AUPRC_repair_slice": temp.sort_values("slice_AUPRC").iloc[0]["slice_name"],
#        "worst_recall_repair_slice": temp.sort_values("slice_recall").iloc[0]["slice_name"],
#    })
#
#repair_eligible_robustness = pd.DataFrame(repair_robust_rows)
#
#display(
#    repair_eligible_robustness.sort_values(
#        "mean_repair_slice_AUPRC",
#        ascending=False
#    )
#)
#
#repair_eligible_robustness.to_csv(
#    "nb9_repair_eligible_slice_robustness.csv",
#    index=False
#)

In [ ]:
# ============================================================
# OFFICIAL FULL-ASAP REPAIR-SLICE ROBUSTNESS
# Uses the governed ASAP pool imported from Notebook 8
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    recall_score,
    precision_score,
    average_precision_score,
)

# ------------------------------------------------------------
# 1. Validate required objects
# ------------------------------------------------------------

required_objects = [
    "df",
    "y",
    "slice_membership",
    "asap_pool",
    "repair_arms",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing required notebook objects: "
        f"{missing_objects}. Run the Notebook 9 loading, "
        "membership, weight-construction and training cells first."
    )

required_asap_columns = [
    "slice_name",
    "asap_eligible_full",
]

missing_asap_columns = [
    column
    for column in required_asap_columns
    if column not in asap_pool.columns
]

if missing_asap_columns:
    raise KeyError(
        "asap_pool is missing required columns: "
        f"{missing_asap_columns}. "
        f"Available columns: {asap_pool.columns.tolist()}"
    )

# ------------------------------------------------------------
# 2. Clean ASAP fields
# ------------------------------------------------------------

asap_eval_pool = asap_pool.copy()

asap_eval_pool["slice_name"] = (
    asap_eval_pool["slice_name"]
    .astype("string")
    .str.strip()
)

asap_eval_pool["asap_eligible_full"] = (
    pd.to_numeric(
        asap_eval_pool["asap_eligible_full"],
        errors="coerce",
    )
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 3. Obtain the official governed evaluation slices
# ------------------------------------------------------------

official_asap_slices = (
    asap_eval_pool.loc[
        asap_eval_pool["asap_eligible_full"].eq(1),
        "slice_name",
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

repair_eligible_slices = [
    slice_name
    for slice_name in official_asap_slices
    if slice_name in slice_membership.columns
]

missing_membership_slices = [
    slice_name
    for slice_name in official_asap_slices
    if slice_name not in slice_membership.columns
]

print(
    "Full-ASAP eligible slices in audit pool:",
    len(official_asap_slices),
)

print(
    "Full-ASAP slices available in membership:",
    len(repair_eligible_slices),
)

print(
    "Eligible slices missing membership:",
    missing_membership_slices,
)

print("\nRepair-evaluation slices:")
print(repair_eligible_slices)

if len(official_asap_slices) == 0:
    raise ValueError(
        "asap_pool contains zero rows with asap_eligible_full == 1. "
        "Return to Notebook 8 and rerun the cell that creates "
        "nb8_asap_complete_candidate_audit.csv."
    )

if len(repair_eligible_slices) == 0:
    raise ValueError(
        "Full-ASAP slices exist, but none match the columns in "
        "slice_membership. Compare slice names for spelling and "
        "underscore differences."
    )

# ------------------------------------------------------------
# 4. Evaluate every trained repair arm
# ------------------------------------------------------------

threshold = 0.5

repair_robust_rows = []
repair_slice_rows = []

y_array = np.asarray(y)

for arm_name in repair_arms.keys():

    probability_col = f"p_{arm_name}"

    if probability_col not in df.columns:
        print(
            f"Skipping {arm_name}: "
            f"{probability_col} is missing."
        )
        continue

    probabilities = (
        pd.to_numeric(
            df[probability_col],
            errors="coerce",
        )
        .to_numpy(dtype=float)
    )

    if not np.isfinite(probabilities).all():
        raise ValueError(
            f"{probability_col} contains NaN or infinite values."
        )

    predictions = (
        probabilities >= threshold
    ).astype(int)

    arm_slice_rows = []

    for slice_name in repair_eligible_slices:

        mask = (
            pd.to_numeric(
                slice_membership[slice_name],
                errors="coerce",
            )
            .fillna(0)
            .astype(bool)
            .to_numpy()
        )

        support = int(mask.sum())

        if support < 30:
            print(
                f"Skipping {slice_name} for {arm_name}: "
                f"support={support}."
            )
            continue

        y_slice = y_array[mask]

        if np.unique(y_slice).size < 2:
            print(
                f"Skipping {slice_name} for {arm_name}: "
                "only one outcome class."
            )
            continue

        row = {
            "arm": arm_name,
            "slice_name": slice_name,
            "support": support,
            "slice_AUPRC": average_precision_score(
                y_slice,
                probabilities[mask],
            ),
            "slice_recall": recall_score(
                y_slice,
                predictions[mask],
                zero_division=0,
            ),
            "slice_precision": precision_score(
                y_slice,
                predictions[mask],
                zero_division=0,
            ),
        }

        arm_slice_rows.append(row)
        repair_slice_rows.append(row)

    # Do not attempt summary calculations on an empty dataframe
    if len(arm_slice_rows) == 0:
        print(
            f"No valid repair slices were available for {arm_name}."
        )
        continue

    temp = pd.DataFrame(arm_slice_rows)

    worst_auprc_row = temp.loc[
        temp["slice_AUPRC"].idxmin()
    ]

    worst_recall_row = temp.loc[
        temp["slice_recall"].idxmin()
    ]

    repair_robust_rows.append({
        "arm": arm_name,

        "n_evaluated_repair_slices":
            len(temp),

        "mean_repair_slice_AUPRC":
            temp["slice_AUPRC"].mean(),

        "worst_repair_slice_AUPRC":
            temp["slice_AUPRC"].min(),

        "avg_to_worst_repair_AUPRC_gap":
            (
                temp["slice_AUPRC"].mean()
                - temp["slice_AUPRC"].min()
            ),

        "mean_repair_slice_recall":
            temp["slice_recall"].mean(),

        "worst_repair_slice_recall":
            temp["slice_recall"].min(),

        "mean_repair_slice_precision":
            temp["slice_precision"].mean(),

        "worst_AUPRC_repair_slice":
            worst_auprc_row["slice_name"],

        "worst_recall_repair_slice":
            worst_recall_row["slice_name"],
    })

# ------------------------------------------------------------
# 5. Build and save outputs
# ------------------------------------------------------------

repair_eligible_slice_results = pd.DataFrame(
    repair_slice_rows
)

repair_eligible_robustness = pd.DataFrame(
    repair_robust_rows
)

if repair_eligible_robustness.empty:
    raise ValueError(
        "No robustness results were produced. Check that the "
        "probability columns exist and that eligible slices have "
        "at least 30 rows and both outcome classes."
    )

display(
    repair_eligible_robustness.sort_values(
        "mean_repair_slice_AUPRC",
        ascending=False,
    )
)

display(
    repair_eligible_slice_results.sort_values(
        ["arm", "slice_AUPRC"],
        ascending=[True, True],
    )
)

repair_eligible_robustness.to_csv(
    "nb9_repair_eligible_slice_robustness.csv",
    index=False,
)

repair_eligible_slice_results.to_csv(
    "nb9_repair_eligible_slice_results.csv",
    index=False,
)

print(
    "\nSaved: nb9_repair_eligible_slice_robustness.csv"
)

print(
    "Saved: nb9_repair_eligible_slice_results.csv"
)

Evaluate the targeted worst slice specifically

In [ ]:
#target_rows = []
#
#mask_target = slice_membership[worst_slice_target].values == 1
#
#for arm_name in repair_arms.keys():
#    prob_col = f"p_{arm_name}"
#
#    y_s = df.loc[mask_target, target_col].values
#    p_s = df.loc[mask_target, prob_col].values
#    pred_s = (p_s >= 0.5).astype(int)
#
#    target_rows.append({
#        "arm": arm_name,
#        "slice": worst_slice_target,
#        "n": int(mask_target.sum()),
#        "AUPRC": average_precision_score(y_s, p_s),
#        "Recall": recall_score(y_s, pred_s),
#        "Precision": precision_score(y_s, pred_s, zero_division=0),
#        "Brier": brier_score_loss(y_s, p_s),
#    })
#
#target_slice_nb9 = pd.DataFrame(target_rows)
#target_slice_nb9

In [ ]:
target_rows = []

if worst_slice_target not in slice_membership.columns:
    print("WARNING: targeted worst slice not found:", worst_slice_target)
    target_slice_nb9 = pd.DataFrame()
else:
    mask_target = slice_membership[worst_slice_target].values == 1

    for arm_name in repair_arms.keys():
        prob_col = f"p_{arm_name}"

        y_s = df.loc[mask_target, target_col].values
        p_s = df.loc[mask_target, prob_col].values
        pred_s = (p_s >= 0.5).astype(int)

        target_rows.append({
            "arm": arm_name,
            "slice": worst_slice_target,
            "n": int(mask_target.sum()),
            "AUPRC": average_precision_score(y_s, p_s),
            "Recall": recall_score(y_s, pred_s),
            "Precision": precision_score(y_s, pred_s, zero_division=0),
            "Brier": brier_score_loss(y_s, p_s),
        })

    target_slice_nb9 = pd.DataFrame(target_rows)

target_slice_nb9

In [ ]:
# ============================================================
# KEY TARGET SLICE RESULTS
# Main clinically meaningful repair targets
# ============================================================

key_target_slices = [
    "abdominal_pain_and_hr_missing",
    "ckd_and_spo2_missing",
    "tachypnea",
    "chest_pain_and_ambulance",
    "hypoxia"
]

key_target_rows = []

for arm_name in repair_arms.keys():
    p = df[f"p_{arm_name}"].values
    pred = (p >= threshold).astype(int)

    for s in key_target_slices:
        if s not in slice_membership.columns:
            continue

        mask = slice_membership[s].astype(bool).values

        if mask.sum() < 30:
            continue

        if len(np.unique(y[mask])) < 2:
            continue

        key_target_rows.append({
            "arm": arm_name,
            "slice": s,
            "n": int(mask.sum()),
            "AUPRC": average_precision_score(y[mask], p[mask]),
            "Recall": recall_score(y[mask], pred[mask], zero_division=0),
            "Precision": precision_score(y[mask], pred[mask], zero_division=0),
            "Brier": brier_score_loss(y[mask], p[mask])
        })

key_target_results = pd.DataFrame(key_target_rows)

display(key_target_results)

key_target_results.to_csv(
    "nb9_key_target_slice_results.csv",
    index=False
)

## Honest Ranker vs Manual Targeted Repair Comparison

In [ ]:
# ============================================================
# HONEST COMPARISON:
# Algorithmic BASiC-Slice ranker vs manual combined_targeted repair
# ============================================================

comparison_arms = [
    "ERM_unweighted",
    "basic_slice_ranked_repair",
    "basic_slice_ranked_top5",
    "basic_slice_ranked_top7",
    "legacy_BASIC_TOP10",
    "combined_targeted"
]

comparison_arms = [a for a in comparison_arms if a in repair_arms]

global_cmp = global_nb9[global_nb9["arm"].isin(comparison_arms)].copy()

repair_cmp = repair_eligible_robustness[
    repair_eligible_robustness["arm"].isin(comparison_arms)
].copy()

main_slice_cmp = key_target_results[
    (key_target_results["arm"].isin(comparison_arms))
    & (key_target_results["slice"] == "abdominal_pain_and_hr_missing")
].copy()

print("GLOBAL COMPARISON")
display(global_cmp)

print("REPAIR-ELIGIBLE SLICE ROBUSTNESS COMPARISON")
display(repair_cmp)

print("MAIN HARMFUL SLICE COMPARISON")
display(main_slice_cmp)

global_cmp.to_csv("nb9_ranker_vs_combined_targeted_global.csv", index=False)
repair_cmp.to_csv("nb9_ranker_vs_combined_targeted_repair_robustness.csv", index=False)
main_slice_cmp.to_csv("nb9_ranker_vs_combined_targeted_main_slice.csv", index=False)

## BASiC-Rank Weight Justification: Feature Importance and Sensitivity

slice-level repair utility table

In [ ]:
# ============================================================
# BUILD SLICE-LEVEL REPAIR UTILITY TARGET
# ERM versus legacy seven-component BASiC TOP10
# ============================================================

import pandas as pd
import numpy as np

baseline_arm = "ERM_unweighted"
repair_arm = "legacy_BASIC_TOP10"

if "revised_slice_results" not in globals():
    raise RuntimeError(
        "Run the unified downstream evaluation cell first."
    )

required_utility_columns = [
    "arm",
    "slice_name",
    "AUPRC",
    "Recall",
    "Brier",
]

missing_utility_columns = [
    column
    for column in required_utility_columns
    if column not in revised_slice_results.columns
]

if missing_utility_columns:
    raise KeyError(
        "revised_slice_results is missing: "
        f"{missing_utility_columns}"
    )

utility_source = (
    revised_slice_results[
        required_utility_columns
    ]
    .copy()
)

baseline = (
    utility_source.loc[
        utility_source["arm"].eq(
            baseline_arm
        )
    ]
    .copy()
)

repair = (
    utility_source.loc[
        utility_source["arm"].eq(
            repair_arm
        )
    ]
    .copy()
)

if baseline.empty:
    raise ValueError(
        f"No rows found for baseline arm: "
        f"{baseline_arm}"
    )

if repair.empty:
    available_arms = sorted(
        utility_source["arm"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    raise ValueError(
        f"No rows found for repair arm: "
        f"{repair_arm}. "
        f"Available arms: {available_arms}"
    )

baseline = (
    baseline.rename(
        columns={
            "slice_name": "slice",
            "AUPRC": "AUPRC_ERM",
            "Recall": "Recall_ERM",
            "Brier": "Brier_ERM",
        }
    )
)

repair = (
    repair.rename(
        columns={
            "slice_name": "slice",
            "AUPRC": "AUPRC_repair",
            "Recall": "Recall_repair",
            "Brier": "Brier_repair",
        }
    )
)

utility = (
    baseline[
        [
            "slice",
            "AUPRC_ERM",
            "Recall_ERM",
            "Brier_ERM",
        ]
    ]
    .merge(
        repair[
            [
                "slice",
                "AUPRC_repair",
                "Recall_repair",
                "Brier_repair",
            ]
        ],
        on="slice",
        how="inner",
        validate="one_to_one",
    )
)

utility["delta_AUPRC"] = (
    utility["AUPRC_repair"]
    - utility["AUPRC_ERM"]
)

utility["delta_Recall"] = (
    utility["Recall_repair"]
    - utility["Recall_ERM"]
)

# Positive values mean lower, improved Brier error.
utility["delta_Brier"] = (
    utility["Brier_ERM"]
    - utility["Brier_repair"]
)

if utility.empty:
    raise ValueError(
        "The ERM and legacy TOP10 results have no "
        "matching repair slices."
    )

print(
    "Utility rows created:",
    len(utility)
)

display(
    utility.sort_values(
        "delta_AUPRC",
        ascending=False,
    )
)

utility.to_csv(
    "nb9_legacy_TOP10_slice_repair_utility.csv",
    index=False,
)

Merge ranker components with repair utility

In [ ]:
# ============================================================
# Merge ranker factors with observed repair utility
# ============================================================

rank_features = rank_table.rename(columns={"slice_name": "slice"})

rank_feature_cols = [
    "slice",
    "harm_norm",
    "dual_source_evidence",
    "clinical_plausibility",
    "stability_score",
    "support_score",
    "artifact_risk",
    "redundancy_risk",
    "basic_slice_rank_score"
]

rank_utility = utility.merge(
    rank_features[rank_feature_cols],
    on="slice",
    how="left"
)

display(rank_utility)

rank_utility.to_csv("nb9_basic_rank_weight_justification_table.csv", index=False)

Correlation-based importance

In [ ]:
# ============================================================
# Correlation between ranker components and repair utility
# ============================================================

component_cols = [
    "harm_norm",
    "dual_source_evidence",
    "clinical_plausibility",
    "stability_score",
    "support_score",
    "artifact_risk",
    "redundancy_risk",
    "basic_slice_rank_score"
]

targets = [
    "delta_AUPRC",
    "delta_Recall",
    "delta_Brier"
]

correlation_rows = []

for target in targets:
    for col in component_cols:
        if rank_utility[col].nunique() <= 1:
            corr = np.nan
        else:
            corr = rank_utility[[col, target]].corr(method="spearman").iloc[0, 1]

        correlation_rows.append({
            "component": col,
            "target": target,
            "spearman_correlation": corr
        })

correlation_importance = pd.DataFrame(correlation_rows)

display(
    correlation_importance.sort_values(
        ["target", "spearman_correlation"],
        ascending=[True, False]
    )
)

correlation_importance.to_csv("nb9_ranker_component_correlations.csv", index=False)

Fit a small interpretable model

In [ ]:
# ============================================================
# SUPPLEMENTARY LEGACY LINEAR UTILITY ANALYSIS
# Run only when enough complete slice-level observations exist.
# ============================================================

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_cols = [
    "harm_norm",
    "dual_source_evidence",
    "clinical_plausibility",
    "stability_score",
    "support_score",
    "artifact_risk",
    "redundancy_risk",
]

required_model_columns = (
    X_cols + ["delta_AUPRC"]
)

missing_model_columns = [
    column
    for column in required_model_columns
    if column not in rank_utility.columns
]

if missing_model_columns:
    raise KeyError(
        "Legacy utility table is missing: "
        f"{missing_model_columns}"
    )

model_data = (
    rank_utility
    .dropna(
        subset=required_model_columns
    )
    .copy()
)

minimum_required_rows = max(
    15,
    2 * len(X_cols)
)

print(
    "Complete rows available:",
    len(model_data)
)

print(
    "Minimum rows required:",
    minimum_required_rows
)

if len(model_data) < minimum_required_rows:
    print(
        "Skipping legacy Ridge analysis because "
        "there are too few complete slice-level "
        "observations for seven predictors."
    )

    coef_table = pd.DataFrame(
        columns=[
            "component",
            "ridge_coefficient_for_delta_AUPRC",
        ]
    )

else:
    X_legacy = model_data[X_cols]
    y_target = model_data[
        "delta_AUPRC"
    ]

    ridge = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            Ridge(alpha=1.0)
        ),
    ])

    ridge.fit(
        X_legacy,
        y_target
    )

    coefficients = (
        ridge
        .named_steps["model"]
        .coef_
    )

    coef_table = (
        pd.DataFrame({
            "component": X_cols,
            "ridge_coefficient_for_delta_AUPRC":
                coefficients,
        })
        .sort_values(
            "ridge_coefficient_for_delta_AUPRC",
            ascending=False,
        )
    )

    display(coef_table)

coef_table.to_csv(
    "nb9_ranker_ridge_coefficients_delta_AUPRC.csv",
    index=False,
)

In [ ]:
# ============================================================
# PRIMARY REVISED THREE-FACTOR RANKER UTILITY ANALYSIS
# ============================================================

revised_rank_table = pd.read_csv(
    "nb8_asap_constrained_basic_rank_table.csv"
)

revised_rank_features = (
    revised_rank_table[
        [
            "slice_name",
            "harm_norm",
            "dual_source_evidence",
            "support_norm_ranker",
            "asap_constrained_basic_score",
        ]
    ]
    .rename(
        columns={
            "slice_name": "slice"
        }
    )
)

revised_rank_utility = (
    utility
    .merge(
        revised_rank_features,
        on="slice",
        how="inner",
    )
)

display(revised_rank_utility)

revised_component_cols = [
    "harm_norm",
    "dual_source_evidence",
    "support_norm_ranker",
    "asap_constrained_basic_score",
]

revised_correlation_rows = []

for target in [
    "delta_AUPRC",
    "delta_Recall",
    "delta_Brier",
]:
    for component in revised_component_cols:
        valid = (
            revised_rank_utility[
                [component, target]
            ]
            .dropna()
        )

        if (
            len(valid) < 3
            or valid[component].nunique() <= 1
            or valid[target].nunique() <= 1
        ):
            correlation = np.nan
        else:
            correlation = (
                valid.corr(
                    method="spearman"
                )
                .iloc[0, 1]
            )

        revised_correlation_rows.append({
            "component": component,
            "target": target,
            "spearman_correlation":
                correlation,
            "n_slices": len(valid),
        })

revised_ranker_correlations = (
    pd.DataFrame(
        revised_correlation_rows
    )
)

display(revised_ranker_correlations)

revised_rank_utility.to_csv(
    "nb9_revised_ranker_utility_table.csv",
    index=False,
)

revised_ranker_correlations.to_csv(
    "nb9_revised_ranker_component_correlations.csv",
    index=False,
)

In [ ]:
if revised_rank_utility.empty:
    utility_slice_names = sorted(
        utility["slice"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    rank_slice_names = sorted(
        revised_rank_features["slice"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    raise ValueError(
        "No revised ranker slices matched the utility table. "
        f"Utility slices: {utility_slice_names}. "
        f"Rank-table slices: {rank_slice_names}."
    )

Permutation importance

In [ ]:
# ============================================================
# PERMUTATION IMPORTANCE FOR LEGACY RANKER COMPONENTS
# ============================================================

from sklearn.inspection import permutation_importance

if (
    "X_legacy" in globals()
    and "y_target" in globals()
    and "ridge" in globals()
    and len(model_data) >= minimum_required_rows
):
    perm = permutation_importance(
        ridge,
        X_legacy,
        y_target,
        n_repeats=100,
        random_state=42,
        scoring="r2",
    )

    perm_table = (
        pd.DataFrame({
            "component": X_cols,
            "permutation_importance_mean":
                perm.importances_mean,
            "permutation_importance_std":
                perm.importances_std,
        })
        .sort_values(
            "permutation_importance_mean",
            ascending=False,
        )
    )

    display(perm_table)

    perm_table.to_csv(
        "nb9_ranker_permutation_importance_delta_AUPRC.csv",
        index=False,
    )

else:
    print(
        "Skipping legacy permutation importance because "
        "the fitted Ridge inputs are unavailable or "
        "there are too few complete slice observations."
    )

    perm_table = pd.DataFrame(
        columns=[
            "component",
            "permutation_importance_mean",
            "permutation_importance_std",
        ]
    )

Sensitivity analysis of weight schemes

In [ ]:
# ============================================================
# Weight sensitivity analysis for BASiC-Rank
# ============================================================
                    #NOW
# ============================================================
# SUPPLEMENTARY LEGACY SEVEN-COMPONENT BASiC SENSITIVITY
# Not the primary ASAP-constrained three-factor ranker
# ============================================================

weight_schemes = {
    "original": {
        "harm_norm": 0.35,
        "dual_source_evidence": 0.20,
        "clinical_plausibility": 0.15,
        "stability_score": 0.15,
        "support_score": 0.10,
        "artifact_risk": -0.25,
        "redundancy_risk": -0.10,
    },

    "harm_heavy": {
        "harm_norm": 0.50,
        "dual_source_evidence": 0.15,
        "clinical_plausibility": 0.10,
        "stability_score": 0.10,
        "support_score": 0.05,
        "artifact_risk": -0.25,
        "redundancy_risk": -0.10,
    },

    "provenance_heavy": {
        "harm_norm": 0.30,
        "dual_source_evidence": 0.35,
        "clinical_plausibility": 0.15,
        "stability_score": 0.10,
        "support_score": 0.05,
        "artifact_risk": -0.25,
        "redundancy_risk": -0.10,
    },

    "artifact_strict": {
        "harm_norm": 0.35,
        "dual_source_evidence": 0.20,
        "clinical_plausibility": 0.15,
        "stability_score": 0.15,
        "support_score": 0.10,
        "artifact_risk": -0.40,
        "redundancy_risk": -0.10,
    },

    "equal_positive": {
        "harm_norm": 0.20,
        "dual_source_evidence": 0.20,
        "clinical_plausibility": 0.20,
        "stability_score": 0.20,
        "support_score": 0.20,
        "artifact_risk": -0.25,
        "redundancy_risk": -0.10,
    },
}

sensitivity_rows = []

for scheme_name, weights in weight_schemes.items():
    temp = rank_table.copy()

    temp[f"rank_score_{scheme_name}"] = 0.0

    for feature, weight in weights.items():
        temp[f"rank_score_{scheme_name}"] += weight * temp[feature]

    selected = (
        temp[
            (temp["nb8_harm"] > 0)
            & (temp["artifact_risk"] == 0)
        ]
        .sort_values(f"rank_score_{scheme_name}", ascending=False)
        .head(10)["slice_name"]
        .tolist()
    )

    overlap_with_original = len(
        set(selected).intersection(set(topk_ranked_slices[10]))
    )

    sensitivity_rows.append({
        "scheme": scheme_name,
        "selected_slices": selected,
        "n_overlap_with_original_TOP10": overlap_with_original,
        "overlap_pct": overlap_with_original / 10
    })

sensitivity_table = pd.DataFrame(sensitivity_rows)

display(sensitivity_table)

sensitivity_table.to_csv("nb9_basic_rank_weight_sensitivity.csv", index=False)

## Bootstrap Confidence Intervals for Key Slice Results

In [ ]:
# ============================================================
# BOOTSTRAP CONFIDENCE INTERVALS
# For main harmful slice and repair-eligible robustness
# ============================================================

from sklearn.metrics import average_precision_score, recall_score, brier_score_loss
import numpy as np
import pandas as pd

RANDOM_STATE = 42
N_BOOT = 500
BOOT_ALPHA = 0.05

rng = np.random.default_rng(RANDOM_STATE)

def bootstrap_slice_metric(y_true, y_prob, threshold=0.5, n_boot=500):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    n = len(y_true)

    auprc_vals = []
    recall_vals = []
    brier_vals = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)

        y_b = y_true[idx]
        p_b = y_prob[idx]

        # Skip bootstrap samples with only one class
        if len(np.unique(y_b)) < 2:
            continue

        pred_b = (p_b >= threshold).astype(int)

        auprc_vals.append(average_precision_score(y_b, p_b))
        recall_vals.append(recall_score(y_b, pred_b, zero_division=0))
        brier_vals.append(brier_score_loss(y_b, p_b))

    def summarize(vals):
        vals = np.asarray(vals)
        return {
            "mean": np.mean(vals),
            "ci_low": np.quantile(vals, BOOT_ALPHA / 2),
            "ci_high": np.quantile(vals, 1 - BOOT_ALPHA / 2),
        }

    return {
        "AUPRC": summarize(auprc_vals),
        "Recall": summarize(recall_vals),
        "Brier": summarize(brier_vals),
    }


ci_arms = [
    "ERM_unweighted",
    "basic_slice_ranked_repair",
    "basic_slice_ranked_top5",
    "basic_slice_ranked_top7",
    "legacy_BASIC_TOP10",
    "combined_targeted",
]
missing_ci_arms = [
    arm_name
    for arm_name in ci_arms
    if f"p_{arm_name}" not in df.columns
]

if missing_ci_arms:
    print(
        "Bootstrap arms without predictions:",
        missing_ci_arms
    )

ci_arms = [
    arm_name
    for arm_name in ci_arms
    if f"p_{arm_name}" in df.columns
]

print(
    "Bootstrap arms retained:",
    ci_arms
)
ci_arms = [a for a in ci_arms if f"p_{a}" in df.columns]

main_slice = "abdominal_pain_and_hr_missing"
mask = slice_membership[main_slice].astype(bool).values

ci_rows = []

for arm_name in ci_arms:
    p = df[f"p_{arm_name}"].values

    ci = bootstrap_slice_metric(
        y_true=y[mask],
        y_prob=p[mask],
        threshold=0.5,
        n_boot=N_BOOT
    )

    ci_rows.append({
        "arm": arm_name,
        "slice": main_slice,

        "AUPRC_mean": ci["AUPRC"]["mean"],
        "AUPRC_ci_low": ci["AUPRC"]["ci_low"],
        "AUPRC_ci_high": ci["AUPRC"]["ci_high"],

        "Recall_mean": ci["Recall"]["mean"],
        "Recall_ci_low": ci["Recall"]["ci_low"],
        "Recall_ci_high": ci["Recall"]["ci_high"],

        "Brier_mean": ci["Brier"]["mean"],
        "Brier_ci_low": ci["Brier"]["ci_low"],
        "Brier_ci_high": ci["Brier"]["ci_high"],
    })

main_slice_bootstrap_ci = pd.DataFrame(ci_rows)

display(main_slice_bootstrap_ci)

main_slice_bootstrap_ci.to_csv(
    "nb9_bootstrap_ci_main_harmful_slice.csv",
    index=False
)

In [ ]:
# ============================================================
# BOOTSTRAP CI FOR WORST REPAIR-ELIGIBLE SLICE
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score


# ------------------------------------------------------------
# Validate prerequisites
# ------------------------------------------------------------

if (
    "official_repair_slices"
    not in globals()
    or not official_repair_slices
):
    raise RuntimeError(
        "Run the finalized ASAP repair-slice "
        "verification cell first."
    )

if (
    "ci_arms"
    not in globals()
    or not ci_arms
):
    raise RuntimeError(
        "Run the main bootstrap confidence-interval "
        "cell first to create ci_arms."
    )

if "bootstrap_slice_metric" not in globals():
    raise RuntimeError(
        "bootstrap_slice_metric is not defined. "
        "Run the preceding bootstrap cell first."
    )


repair_eligible_slices = list(
    official_repair_slices
)

worst_ci_rows = []


# ------------------------------------------------------------
# Identify each arm's observed worst governed slice
# and estimate bootstrap confidence intervals
# ------------------------------------------------------------

for arm_name in ci_arms:

    prediction_column = f"p_{arm_name}"

    if prediction_column not in df.columns:
        print(
            "Skipping missing prediction column:",
            prediction_column,
        )
        continue

    p = (
        df[prediction_column]
        .to_numpy(dtype=float)
    )

    observed_rows = []

    for slice_name in repair_eligible_slices:

        if slice_name not in slice_membership.columns:
            continue

        mask_slice = (
            slice_membership[slice_name]
            .astype(bool)
            .to_numpy()
        )

        slice_support = int(
            mask_slice.sum()
        )

        if slice_support < 30:
            continue

        if (
            np.unique(
                y[mask_slice]
            ).size
            < 2
        ):
            continue

        observed_rows.append({
            "slice": slice_name,
            "support": slice_support,
            "AUPRC": average_precision_score(
                y[mask_slice],
                p[mask_slice],
            ),
        })

    observed_df = pd.DataFrame(
        observed_rows
    )

    if observed_df.empty:
        print(
            "No eligible slices available for:",
            arm_name,
        )
        continue

    worst_observed_row = (
        observed_df
        .sort_values(
            "AUPRC",
            ascending=True,
        )
        .iloc[0]
    )

    worst_slice = str(
        worst_observed_row[
            "slice"
        ]
    )

    mask_worst = (
        slice_membership[worst_slice]
        .astype(bool)
        .to_numpy()
    )

    ci = bootstrap_slice_metric(
        y_true=y[mask_worst],
        y_prob=p[mask_worst],
        threshold=0.5,
        n_boot=N_BOOT,
    )

    worst_ci_rows.append({
        "arm": arm_name,
        "worst_repair_slice": worst_slice,
        "worst_slice_support": int(
            mask_worst.sum()
        ),

        "observed_worst_slice_AUPRC":
            float(
                worst_observed_row[
                    "AUPRC"
                ]
            ),

        "worst_slice_AUPRC_mean":
            ci["AUPRC"]["mean"],

        "worst_slice_AUPRC_ci_low":
            ci["AUPRC"]["ci_low"],

        "worst_slice_AUPRC_ci_high":
            ci["AUPRC"]["ci_high"],

        "worst_slice_Recall_mean":
            ci["Recall"]["mean"],

        "worst_slice_Recall_ci_low":
            ci["Recall"]["ci_low"],

        "worst_slice_Recall_ci_high":
            ci["Recall"]["ci_high"],

        "worst_slice_Brier_mean":
            ci["Brier"]["mean"],

        "worst_slice_Brier_ci_low":
            ci["Brier"]["ci_low"],

        "worst_slice_Brier_ci_high":
            ci["Brier"]["ci_high"],
    })


# ------------------------------------------------------------
# Create dataframe only after rows have been collected
# ------------------------------------------------------------

worst_repair_slice_bootstrap_ci = pd.DataFrame(
    worst_ci_rows
)

if worst_repair_slice_bootstrap_ci.empty:
    raise ValueError(
        "No worst-slice bootstrap results were produced. "
        "Check ci_arms, official_repair_slices, "
        "prediction columns, and slice memberships."
    )

display(
    worst_repair_slice_bootstrap_ci
)

worst_repair_slice_bootstrap_ci.to_csv(
    "nb9_bootstrap_ci_worst_repair_slice.csv",
    index=False,
)

print(
    "Worst repair-slice bootstrap results saved:",
    len(
        worst_repair_slice_bootstrap_ci
    ),
    "arms",
)

In [ ]:
# ============================================================
# THRESHOLD SENSITIVITY FOR MAIN TARGET SLICE
# Shows whether recall loss is threshold-related
# ============================================================

main_slice = "abdominal_pain_and_hr_missing"
threshold_grid = np.arange(0.10, 0.91, 0.05)

threshold_rows = []

mask = slice_membership[main_slice].astype(bool).values

threshold_sensitivity_arms = [
    "ERM_unweighted",
    "legacy_BASIC_TOP10",
    "revised_P1_provenance_rank_only",
    "revised_P2_provenance_weight_only",
    "revised_P3_provenance_both",
    "revised_P4_no_provenance",
    "combined_targeted",
    "dual_supported_repair",
    "dual_source_combined_discovery_repair",
]

threshold_sensitivity_arms = [
    arm_name
    for arm_name in threshold_sensitivity_arms
    if f"p_{arm_name}" in df.columns
]

for arm_name in threshold_sensitivity_arms:
    p = df[f"p_{arm_name}"].values

    for t in threshold_grid:
        pred = (p >= t).astype(int)

        threshold_rows.append({
            "arm": arm_name,
            "slice": main_slice,
            "threshold": t,
            "Recall": recall_score(y[mask], pred[mask], zero_division=0),
            "Precision": precision_score(y[mask], pred[mask], zero_division=0)
        })

threshold_sensitivity = pd.DataFrame(threshold_rows)

display(threshold_sensitivity)

threshold_sensitivity.to_csv(
    "nb9_threshold_sensitivity_abdominal_pain_hr_missing.csv",
    index=False
)

## Recall-Preserving Threshold Recalibration

In [ ]:
# ============================================================
# RECALL-PRESERVING THRESHOLD RECALIBRATION
# Goal: recover ERM recall on the main harmful slice
# ============================================================

from sklearn.metrics import recall_score, precision_score, average_precision_score, brier_score_loss

main_slice = "abdominal_pain_and_hr_missing"
mask = slice_membership[main_slice].astype(bool).values

threshold_grid = np.arange(0.05, 0.951, 0.01)

arms_to_check = [
    "ERM_unweighted",
    "basic_slice_ranked_repair",
    "basic_slice_ranked_top5",
    "basic_slice_ranked_top7",
    "legacy_BASIC_TOP10",
    "dual_supported_repair",
    "dual_source_combined_discovery_repair",
    "combined_targeted",
]
missing_threshold_arms = [
    arm_name
    for arm_name in arms_to_check
    if f"p_{arm_name}" not in df.columns
]

if missing_threshold_arms:
    print(
        "Threshold arms without predictions:",
        missing_threshold_arms
    )

arms_to_check = [
    arm_name
    for arm_name in arms_to_check
    if f"p_{arm_name}" in df.columns
]

print(
    "Threshold arms retained:",
    arms_to_check
)
arms_to_check = [a for a in arms_to_check if f"p_{a}" in df.columns]

# Baseline recall at threshold 0.5
erm_p = df[f"p_ERM_unweighted"].values
erm_pred_05 = (erm_p >= 0.5).astype(int)
erm_recall_05 = recall_score(y[mask], erm_pred_05[mask], zero_division=0)

TARGET_RECALL = erm_recall_05

print("Main slice:", main_slice)
print("ERM recall at threshold 0.5:", TARGET_RECALL)

threshold_selection_rows = []

for arm_name in arms_to_check:
    p = df[f"p_{arm_name}"].values

    candidates = []

    for t in threshold_grid:
        pred = (p >= t).astype(int)

        r = recall_score(y[mask], pred[mask], zero_division=0)
        pr = precision_score(y[mask], pred[mask], zero_division=0)

        candidates.append({
            "arm": arm_name,
            "threshold": t,
            "Recall": r,
            "Precision": pr,
            "AUPRC": average_precision_score(y[mask], p[mask]),
            "Brier": brier_score_loss(y[mask], p[mask])
        })

    cand = pd.DataFrame(candidates)

    feasible = cand[cand["Recall"] >= TARGET_RECALL].copy()

    if len(feasible) > 0:
        # Among thresholds that preserve recall, choose the one with best precision
        best = feasible.sort_values(
            ["Precision", "threshold"],
            ascending=[False, False]
        ).iloc[0]
    else:
        # If no threshold reaches target recall, choose max recall
        best = cand.sort_values(
            ["Recall", "Precision"],
            ascending=[False, False]
        ).iloc[0]

    threshold_selection_rows.append(best.to_dict())

threshold_recalibration_results = pd.DataFrame(threshold_selection_rows)

display(
    threshold_recalibration_results.sort_values(
        ["Recall", "Precision"],
        ascending=False
    )
)

threshold_recalibration_results.to_csv(
    "nb9_recall_preserving_threshold_recalibration.csv",
    index=False
)

## Final Compact Reporting Table

In [ ]:
## ============================================================
## FINAL COMPACT REPORTING TABLE
## ERM vs Official Ranker TOP_K=10 vs Manual Combined Targeted
## ============================================================
#
#final_arms = [
#    "ERM_unweighted",
#    "basic_slice_ranked_top10",
#    "combined_targeted"
#]
#
## Global metrics
#global_final = global_cmp[global_cmp["arm"].isin(final_arms)].copy()
#
## Repair robustness metrics
#repair_final = repair_cmp[repair_cmp["arm"].isin(final_arms)].copy()
#
## Main harmful slice metrics at threshold 0.5
#main_final = main_slice_cmp[main_slice_cmp["arm"].isin(final_arms)].copy()
#
## Threshold recalibration metrics
#threshold_final = threshold_recalibration_results[
#    threshold_recalibration_results["arm"].isin(final_arms)
#].copy()
#
#rows = []
#
#for arm in final_arms:
#    g = global_final[global_final["arm"] == arm].iloc[0]
#    r = repair_final[repair_final["arm"] == arm].iloc[0]
#    m = main_final[main_final["arm"] == arm].iloc[0]
#    t = threshold_final[threshold_final["arm"] == arm].iloc[0]
#
#    rows.append({
#        "arm": arm,
#
#        "global_AUPRC": g["AUPRC"],
#        "global_AUROC": g["AUROC"],
#        "global_Brier": g["Brier"],
#
#        "mean_repair_slice_AUPRC": r["mean_repair_slice_AUPRC"],
#        "worst_repair_slice_AUPRC": r["worst_repair_slice_AUPRC"],
#        "repair_AUPRC_gap": r["avg_to_worst_repair_AUPRC_gap"],
#
#        "main_slice_AUPRC": m["AUPRC"],
#        "main_slice_recall_at_0_5": m["Recall"],
#        "main_slice_precision_at_0_5": m["Precision"],
#        "main_slice_Brier": m["Brier"],
#
#        "recalibrated_threshold": t["threshold"],
#        "main_slice_recalibrated_recall": t["Recall"],
#        "main_slice_recalibrated_precision": t["Precision"],
#    })
#
#final_compact_report_table = pd.DataFrame(rows)
#
#display(final_compact_report_table)
#
#final_compact_report_table.to_csv(
#    "nb9_final_compact_report_table.csv",
#    index=False
#)
#
#print("Saved: nb9_final_compact_report_table.csv")

Compare percentage upweighted

In [ ]:
# Do not overwrite repair_arms here.
# Keep the repair_arms dictionary created in the training cell.

print("Repair arms still active:")
print(list(repair_arms.keys()))

In [ ]:
weight_summary_rows = []

for arm_name, weight_col in repair_arms.items():

    if weight_col is None:
        w = np.ones(len(df))
    else:
        w = df[weight_col].values

    weight_summary_rows.append({
        "arm": arm_name,
        "weight_col": weight_col if weight_col is not None else "None_baseline",
        "mean_weight": np.mean(w),
        "std_weight": np.std(w),
        "min_weight": np.min(w),
        "max_weight": np.max(w),
        "percent_upweighted": 100 * (w > 1.001).mean(),
        "percent_downweighted": 100 * (w < 0.999).mean(),
        "percent_changed": 100 * (np.abs(w - 1) > 0.001).mean(),
    })

weight_summary_nb9 = pd.DataFrame(weight_summary_rows)
display(weight_summary_nb9)

weight_summary_nb9.to_csv("nb9_weight_summary_all_repair_arms.csv", index=False)

Decision table: choose best arm

In [ ]:
decision_table = (
    global_nb9
    .merge(robust_nb9, on="arm", how="left")
    .merge(weight_summary_nb9[["arm", "percent_upweighted", "percent_changed"]], on="arm", how="left")
)

decision_table.sort_values(
    ["worst_slice_recall", "avg_to_worst_recall_gap", "AUPRC"],
    ascending=[False, True, False]
)

In [ ]:
# ============================================================
# FINAL ARM-COVERAGE AUDIT
# ============================================================

result_tables_to_check = {
    "global":
        revised_global_results,

    "repair_robustness":
        revised_repair_robustness,

    "main_slice":
        revised_slice_results.loc[
            revised_slice_results[
                "slice_name"
            ].eq(
                "abdominal_pain_and_hr_missing"
            )
        ],

    "threshold":
        revised_threshold_recalibration,
}

coverage_rows = []

for table_name, table in (
    result_tables_to_check.items()
):
    available = (
        set(
            table["arm"]
            .dropna()
            .astype(str)
        )
        if "arm" in table.columns
        else set()
    )

    coverage_rows.append({
        "result_table": table_name,
        "legacy_BASIC_TOP10_present":
            ARM_LEGACY_TOP10 in available,
        "n_arms": len(available),
    })

arm_coverage_audit = pd.DataFrame(
    coverage_rows
)

display(arm_coverage_audit)

required_legacy_tables = [
    "global",
    "repair_robustness",
    "main_slice",
    "threshold",
]

missing_legacy_tables = (
    arm_coverage_audit.loc[
        arm_coverage_audit[
            "result_table"
        ].isin(
            required_legacy_tables
        )
        &
        ~arm_coverage_audit[
            "legacy_BASIC_TOP10_present"
        ],
        "result_table"
    ]
    .tolist()
)

if missing_legacy_tables:
    raise ValueError(
        "legacy_BASIC_TOP10 is missing from: "
        f"{missing_legacy_tables}"
    )

In [ ]:
# ============================================================
# FINAL PRIMARY ARM-COVERAGE AUDIT
# Only the official primary comparison belongs in the main table
# ============================================================

required_primary_tables = {
    "global_results":
        revised_global_results,

    "repair_robustness":
        revised_repair_robustness,

    "main_slice_results":
        revised_slice_results.loc[
            revised_slice_results[
                "slice_name"
            ].eq(main_slice)
        ],

    "threshold_recalibration":
        revised_threshold_recalibration,

    "final_report":
        final_asap_constrained_report,
}

required_primary_arms = (
    PRIMARY_METHOD_ARMS.copy()
)

coverage_rows = []

for table_name, table in required_primary_tables.items():

    available_arms = set(
        table["arm"]
        .dropna()
        .astype(str)
        .tolist()
    )

    for arm_name in required_primary_arms:
        coverage_rows.append({
            "result_table":
                table_name,

            "arm":
                arm_name,

            "present":
                arm_name in available_arms,
        })

primary_arm_coverage = pd.DataFrame(
    coverage_rows
)

display(primary_arm_coverage)

missing_primary_coverage = (
    primary_arm_coverage.loc[
        ~primary_arm_coverage[
            "present"
        ]
    ]
)

if not missing_primary_coverage.empty:
    display(
        missing_primary_coverage
    )

    raise ValueError(
        "One or more official primary arms are "
        "missing from required result tables."
    )

print(
    "All official primary reporting arms are present."
)

In [ ]:
# ============================================================
# PROVENANCE-ABLATION COVERAGE AUDIT
# ============================================================

required_ablation_tables = {
    "global_results":
        revised_global_results,

    "repair_robustness":
        revised_repair_robustness,

    "provenance_ablation_report":
        provenance_ablation_report,
}

ablation_coverage_rows = []

for table_name, table in (
    required_ablation_tables.items()
):

    available_arms = set(
        table["arm"]
        .dropna()
        .astype(str)
        .tolist()
    )

    for arm_name in (
        PROVENANCE_ABLATION_ARMS
    ):
        ablation_coverage_rows.append({
            "result_table":
                table_name,

            "arm":
                arm_name,

            "present":
                arm_name in available_arms,
        })

provenance_ablation_coverage = (
    pd.DataFrame(
        ablation_coverage_rows
    )
)

display(
    provenance_ablation_coverage
)

missing_ablation_coverage = (
    provenance_ablation_coverage.loc[
        ~provenance_ablation_coverage[
            "present"
        ]
    ]
)

if not missing_ablation_coverage.empty:
    display(
        missing_ablation_coverage
    )

    raise ValueError(
        "One or more P1–P4 ablation arms are missing."
    )

print(
    "All P1–P4 provenance-ablation arms are present."
)

Save outputs

In [ ]:
#lobal_nb9.to_csv("nb9_global_results.csv", index=False)
#obust_nb9.to_csv("nb9_slice_robustness_results.csv", index=False)
#arget_slice_nb9.to_csv("nb9_target_slice_results.csv", index=False)
#eight_summary_nb9.to_csv("nb9_weight_summary.csv", index=False)
#ecision_table.to_csv("nb9_decision_table.csv", index=False)
#
#or arm_name, sr in slice_results_dict.items():
#   sr.to_csv(f"nb9_slice_results__{arm_name}.csv", index=False)
#
#rint("Notebook 9 outputs saved.")

In [ ]:
# ============================================================
# DUAL-SOURCE COMPARISON COVERAGE AUDIT
# ============================================================

required_dual_source_arms = [
    "ERM_unweighted",
    "clinical_discovery_repair",
    "error_guided_discovery_repair",
    "dual_supported_repair",
    "dual_source_combined_discovery_repair",
]

dual_source_tables = {
    "global_results": global_nb9,
    "robustness_results": robust_nb9,
    "decision_table": decision_table,
}

dual_source_coverage_rows = []

for table_name, table in dual_source_tables.items():

    available_arms = set(
        table["arm"]
        .dropna()
        .astype(str)
    )

    for arm_name in required_dual_source_arms:
        dual_source_coverage_rows.append({
            "table": table_name,
            "arm": arm_name,
            "present":
                arm_name in available_arms,
        })

dual_source_arm_coverage = pd.DataFrame(
    dual_source_coverage_rows
)

display(
    dual_source_arm_coverage
)

missing_dual_source_coverage = (
    dual_source_arm_coverage.loc[
        ~dual_source_arm_coverage[
            "present"
        ]
    ]
)

if not missing_dual_source_coverage.empty:
    display(
        missing_dual_source_coverage
    )

    raise ValueError(
        "One or more dual-source comparison arms "
        "are missing from required result tables."
    )

print(
    "All dual-source comparison arms are present "
    "in all required result tables."
)

In [ ]:
dual_source_arm_names = [
    "ERM_unweighted",
    "clinical_discovery_repair",
    "error_guided_discovery_repair",
    "dual_supported_repair",
    "dual_source_combined_discovery_repair",
]

In [ ]:
global_nb9.to_csv("nb9_global_results.csv", index=False)
robust_nb9.to_csv("nb9_slice_robustness_results.csv", index=False)
target_slice_nb9.to_csv("nb9_target_slice_results.csv", index=False)
weight_summary_nb9.to_csv("nb9_weight_summary.csv", index=False)
decision_table.to_csv("nb9_decision_table.csv", index=False)

# Special extract: the actual proof table for the dual-source discovery claim
dual_source_arm_names = [
    "ERM_unweighted",
    "clinical_discovery_repair",
    "error_guided_discovery_repair",
    "dual_source_combined_discovery_repair",
]
missing_dual_source_arms = [
    arm_name
    for arm_name in dual_source_arm_names
    if arm_name not in decision_table["arm"].values
]

if missing_dual_source_arms:
    print(
        "Dual-source comparison arms missing:",
        missing_dual_source_arms
    )
dual_source_repair_comparison = decision_table[
    decision_table["arm"].isin(dual_source_arm_names)
].copy()

dual_source_repair_comparison.to_csv(
    "nb9_dual_source_repair_comparison.csv", index=False
)

for arm_name, sr in slice_results_dict.items():
    sr.to_csv(f"nb9_slice_results__{arm_name}.csv", index=False)

print("Notebook 9 outputs saved.")
print("- nb9_global_results.csv")
print("- nb9_slice_robustness_results.csv")
print("- nb9_target_slice_results.csv")
print("- nb9_weight_summary.csv")
print("- nb9_decision_table.csv")
print("- nb9_dual_source_repair_comparison.csv")